# Neutralino Decay and Fine-Tuning in the pMSSM

In [ ]:
import sys
print(sys.version)
print(sys.executable)

import numpy as np
import pandas as pd
from pathlib import Path
import os
import uproot

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.stats import gaussian_kde
from scipy.stats import binned_statistic_2d
import matplotlib.colors as mcolors
from matplotlib.colors import LogNorm
import seaborn as sns
from tqdm import tqdm

import tarfile
import re

In [ ]:
ID = 5291650
datapath = Path(f'../run/{ID}').expanduser()

print(f'Loading data from {datapath}')

if datapath.exists():
    print([f.name for f in datapath.iterdir()])
else:
    print("Directory not found!")


In [ ]:
FIGURES_DIR = "../outputs/figures"

# Ensure the directory exists before saving anything
os.makedirs(FIGURES_DIR, exist_ok=True)

def save_fig(filename, dpi=300, bbox_inches='tight', **kwargs):
    """Automatically saves the current figure to pMSSM-Analysis/outputs/figures/"""
    filepath = os.path.join(FIGURES_DIR, filename)
    plt.savefig(filepath, dpi=dpi, bbox_inches=bbox_inches, **kwargs)
    print(f"Figure saved successfully to: {filepath}")

In [ ]:
dfs = []

for d in sorted(
    [p for p in datapath.iterdir() if p.is_dir()],
    key=lambda x: int(x.name)
):
    dir_num = d.name
    root_file = d / f"ntuple.{ID}.{dir_num}.root"
    if not root_file.exists():
        continue

    with uproot.open(root_file) as f:
        tname = f.keys()[0]
        tree = f[tname]
        df = tree.arrays(library="pd")
        df["dir"] = int(dir_num)
        df["source_file"] = root_file.name
        cuts = (df['SS_m_h']!=-1) & (df['SP_m_h']!=-1) & (df['MO_Omega']!=-1) &\
        (df['SI_BR_Bs_to_mumu']!=-1) & (df['GM2_gmuon']!=-1) & (df['FH_m_h']!=-1) #& (df['FH_m_h']>122) & (df['FH_m_h']<128)
        df = df[cuts]
        dfs.append(df)

full_data = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

In [ ]:
# print(full_data)
full_data.info()

In [ ]:
# full_data

In [ ]:
# # list all column names in the dataframe
# for col in full_data.columns:
#     print(col)

In [ ]:
def extract_h_decay_brs(slha_content):
    """
    Extract two-body branching ratios of the light Higgs (h^0, PDG 25)
    from the DECAY 25 block. Only NDA==2 rows with an exact daughter-ID
    match are accepted, so three-body modes (W^+ s c_bar, Z s s_bar, etc.)
    are never summed into the two-body channels.
    Returns a dict of BRs.
    """
    lines = slha_content.decode('utf-8').splitlines()

    # two-body channels we want, keyed by the exact daughter-ID set
    targets = {
        frozenset({3, -3}):   'h_to_s_s',
        frozenset({5, -5}):   'h_to_b_b',
        frozenset({4, -4}):   'h_to_c_c',
        frozenset({15, -15}): 'h_to_tau_tau',
        frozenset({13, -13}): 'h_to_mu_mu',
        frozenset({21}):      'h_to_g_g',     # g g -> {21} as a set
        frozenset({22}):      'h_to_gam_gam', # gamma gamma -> {22}
    }
    brs = {name: 0.0 for name in targets.values()}

    in_h_block = False
    for line in lines:
        # Enter the h^0 decay block (PDG 25), leave on the next DECAY.
        m = re.match(r'^\s*DECAY\s+(\d+)', line, re.IGNORECASE)
        if m:
            in_h_block = (m.group(1) == '25')
            continue
        if not in_h_block:
            continue

        stripped = line.strip()
        if stripped.startswith('#') or not stripped:
            continue

        parts = stripped.split()
        # two-body rows: BR, NDA(=2), ID1, ID2  -> 4 tokens before the comment
        if len(parts) >= 4 and parts[1] == '2':
            try:
                ids = frozenset({int(parts[2]), int(parts[3])})
            except ValueError:
                continue
            if ids in targets:
                brs[targets[ids]] = float(parts[0])

    return brs

In [ ]:
def extract_slha_params(slha_content):
    """Extract alpha, mHd2, mHu2, y_t, y_b, and Q_susy from SLHA file content."""
    lines = slha_content.decode('utf-8').splitlines()
    
    alpha = None
    mHd2 = None
    mHu2 = None
    y_t = None
    y_b = None
    Q_susy = None
    
    in_alpha_block = False
    in_msoft_block = False
    in_yu_block = False
    in_yd_block = False
    
    for line in lines:
        # --- Block headers ---
        if re.match(r'^\s*block\s+alpha', line, re.IGNORECASE):
            in_alpha_block = True
            in_msoft_block = in_yu_block = in_yd_block = False
            continue
        
        if re.match(r'^\s*block\s+msoft', line, re.IGNORECASE):
            in_msoft_block = True
            in_alpha_block = in_yu_block = in_yd_block = False
            continue
        
        if re.match(r'^\s*block\s+yu', line, re.IGNORECASE):
            in_yu_block = True
            in_alpha_block = in_msoft_block = in_yd_block = False
            # Extract Q from header
            q_match = re.search(r'Q=\s*([\d.eE+\-]+)', line, re.IGNORECASE)
            if q_match:
                Q_susy = float(q_match.group(1))
            continue
        
        if re.match(r'^\s*block\s+yd', line, re.IGNORECASE):
            in_yd_block = True
            in_alpha_block = in_msoft_block = in_yu_block = False
            continue
        
        if re.match(r'^\s*block', line, re.IGNORECASE):
            in_alpha_block = in_msoft_block = in_yu_block = in_yd_block = False
            continue
        
        stripped = line.strip()
        if stripped.startswith('#') or not stripped:
            continue
        
        # --- Extract alpha ---
        if in_alpha_block:
            match = re.match(r'\s*([-+]?\d+\.?\d*[eE]?[-+]?\d*)', stripped)
            if match:
                alpha = float(match.group(1))
                in_alpha_block = False
        
        # --- Extract mHd2 and mHu2 from MSOFT ---
        if in_msoft_block:
            parts = stripped.split()
            if len(parts) >= 2:
                if parts[0] == '21':
                    mHd2 = float(parts[1])
                elif parts[0] == '22':
                    mHu2 = float(parts[1])
        
        # --- Extract y_t from Yu block (entry 3 3) ---
        if in_yu_block:
            parts = stripped.split()
            if len(parts) >= 3 and parts[0] == '3' and parts[1] == '3':
                y_t = float(parts[2])
        
        # --- Extract y_b from Yd block (entry 3 3) ---
        if in_yd_block:
            parts = stripped.split()
            if len(parts) >= 3 and parts[0] == '3' and parts[1] == '3':
                y_b = float(parts[2])
    
    return alpha, mHd2, mHu2, y_t, y_b, Q_susy


def get_params_for_model(datapath, ID, dir_num, model_num):
    """Extract all parameters for a single model."""
    local_model = int(model_num % 100000)
    tarball = datapath / str(dir_num) / f'scan.{ID}.{dir_num}.tar.gz'
    slha_path = f'scan.{ID}.{dir_num}/SPheno/{local_model}.slha'
    
    try:
        with tarfile.open(tarball, 'r:gz') as tar:
            try:
                f = tar.extractfile(slha_path)
                if f is None:
                    return None, None, None, None, None, None
                return extract_slha_params(f.read())
            except KeyError:
                return None, None, None, None, None, None
    except Exception:
        return None, None, None, None, None, None


In [ ]:
results = []

valid_dirs = [d for d in range(100)
              if (datapath / str(d) / f'scan.{ID}.{d}.tar.gz').exists()]

for dir_num in tqdm(valid_dirs, desc='Processing directories'):
    tar_path = datapath / str(dir_num) / f'scan.{ID}.{dir_num}.tar.gz'
    dir_models = full_data[full_data['dir'] == dir_num]
    if len(dir_models) == 0:
        continue

    with tarfile.open(tar_path, 'r:gz') as tar:
        for _, row in dir_models.iterrows():
            local_model = int(row['model'] % 100000)
            slha_path = f'scan.{ID}.{dir_num}/SPheno/{local_model}.slha'
            try:
                f = tar.extractfile(slha_path)
                if f is None:
                    results.append((row.name, None, None, None, None, None, None, None))
                    continue
                content = f.read()
                alpha, mHd2, mHu2, y_t, y_b, Q_susy = extract_slha_params(content)
                h_brs = extract_h_decay_brs(content)   # <-- new
                results.append((row.name, alpha, mHd2, mHu2, y_t, y_b, Q_susy, h_brs))
            except KeyError:
                results.append((row.name, None, None, None, None, None, None, None))

idx, alpha_vals, mHd2_vals, mHu2_vals, yt_vals, yb_vals, Q_vals, hbr_vals = zip(*results)
full_data.loc[list(idx), 'alpha']  = alpha_vals
full_data.loc[list(idx), 'mHd2']   = mHd2_vals
full_data.loc[list(idx), 'mHu2']   = mHu2_vals
full_data.loc[list(idx), 'y_t']    = yt_vals
full_data.loc[list(idx), 'y_b']    = yb_vals
full_data.loc[list(idx), 'Q_susy'] = Q_vals

print(f"alpha found for {full_data['alpha'].notna().sum()} / {len(full_data)} models")
print(f"mHd2 found for {full_data['mHd2'].notna().sum()} / {len(full_data)} models")
print(f"mHu2 found for {full_data['mHu2'].notna().sum()} / {len(full_data)} models")
print(f"y_t found for {full_data['y_t'].notna().sum()} / {len(full_data)} models")
print(f"y_b found for {full_data['y_b'].notna().sum()} / {len(full_data)} models")
print(f"Q_susy found for {full_data['Q_susy'].notna().sum()} / {len(full_data)} models")

# Overwrite the existing (contaminated) SP_BF_h_to_* columns with clean values.
for col_suffix in ['s_s', 'b_b', 'c_c', 'tau_tau', 'mu_mu', 'g_g', 'gam_gam']:
    full_data.loc[list(idx), f'SP_BF_h_to_{col_suffix}'] = [
        (d.get(f'h_to_{col_suffix}') if isinstance(d, dict) else None) for d in hbr_vals
    ]

In [ ]:
print("s_s mean:", full_data['SP_BF_h_to_s_s'].mean())   # expect ~1e-4, NOT ~0.08
print("tau mean:", full_data['SP_BF_h_to_tau_tau'].mean()) # expect ~0.089, was 0.106
print("bb  mean:", full_data['SP_BF_h_to_b_b'].mean())     # expect ~0.560, was 0.534

In [ ]:
# --- Compute Delta_EW with stop and sbottom loop corrections ---
mZ = 91.1876
v = 246.0

tanb = full_data['SP_tanb_min']
tanb2 = tanb ** 2

sin2_tW = 1 - (full_data['SP_m_W'] / mZ)**2
x_W = sin2_tW
cos_2beta = (1 - tanb**2) / (1 + tanb**2)
g_Z2 = mZ**2 / (2 * v**2)

full_data['tan_tW'] = np.sqrt((mZ**2 - full_data['SP_m_W']**2) / full_data['SP_m_W']**2)

# Tree-level terms
C_mu  = full_data['SP_mu'] ** 2
C_mHu = (full_data['mHu2'] * tanb2 / (tanb2 - 1)).abs()
C_mHd = (full_data['mHd2'] / (tanb2 - 1)).abs()

# Stop loop contribution to Sigma_u^u
m_st1 = full_data['SP_m_t_1'].abs()
m_st2 = full_data['SP_m_t_2'].abs()
y_t   = full_data['y_t']
Q     = full_data['Q_susy']
mu    = full_data['SP_mu'].abs()
A_t   = full_data['SP_At']

m_tL2   = full_data['SP_mqL3']**2
m_tR2   = full_data['SP_mtR']**2
Delta_t = (m_tL2 - m_tR2) / 2 + mZ**2 * cos_2beta * (0.25 - 2*x_W/3)

def F(m2, Q):
    return m2 * (np.log(m2 / Q**2) - 1)

def stop_loop(m_st, sign, y_t, g_Z2, A_t, Delta_t, m_st1, m_st2, Q):
    num = y_t**2 * A_t**2 - 8 * g_Z2 * (0.25 - 2*x_W/3) * Delta_t
    denom = m_st2**2 - m_st1**2
    bracket = y_t**2 - g_Z2 + sign * num / denom
    return (3 / (16 * np.pi**2)) * F(m_st**2, Q) * bracket

Sigma_uu = (
    stop_loop(m_st1, -1, y_t, g_Z2, A_t, Delta_t, m_st1, m_st2, Q) +
    stop_loop(m_st2, +1, y_t, g_Z2, A_t, Delta_t, m_st1, m_st2, Q)
)
C_Sigma_uu = (Sigma_uu * tanb2 / (tanb2 - 1)).abs()

# Sbottom loop contribution to Sigma_d^d
m_sb1 = full_data['SP_m_b_1'].abs()
m_sb2 = full_data['SP_m_b_2'].abs()
y_b   = full_data['y_b']
A_b   = full_data['SP_Ab']

m_bL2   = full_data['SP_mqL3']**2
m_bR2   = full_data['SP_mbR']**2
Delta_b = (m_bL2 - m_bR2) / 2 - mZ**2 * cos_2beta * (0.25 - x_W/3)

def sbottom_loop(m_sb, sign, y_b, g_Z2, A_b, Delta_b, m_sb1, m_sb2, Q):
    num = y_b**2 * A_b**2 - 8 * g_Z2 * (0.25 - x_W/3) * Delta_b
    denom = m_sb2**2 - m_sb1**2
    bracket = y_b**2 - g_Z2 + sign * num / denom
    return (3 / (16 * np.pi**2)) * F(m_sb**2, Q) * bracket

Sigma_dd = (
    sbottom_loop(m_sb1, -1, y_b, g_Z2, A_b, Delta_b, m_sb1, m_sb2, Q) +
    sbottom_loop(m_sb2, +1, y_b, g_Z2, A_b, Delta_b, m_sb1, m_sb2, Q)
)
C_Sigma_dd = (Sigma_dd / (tanb2 - 1)).abs()

# Full Delta_EW
full_data['Delta_EW'] = (
    np.maximum.reduce([C_mu, C_mHu, C_mHd, C_Sigma_uu, C_Sigma_dd])
) / (mZ**2 / 2)

full_data['C_mu'] = C_mu.abs()
full_data['C_mHu'] = C_mHu.abs()
full_data['C_mHd'] = C_mHd.abs()
full_data['C_Sigma_uu'] = C_Sigma_uu.abs()
full_data['C_Sigma_dd'] = C_Sigma_dd.abs()

print(f"\nDelta_EW computed for {full_data['Delta_EW'].notna().sum()} models")
print(full_data['Delta_EW'].describe())

In [ ]:
# Apply the Higgs mass filter to your valid dataset
# df_analysis = full_data[(full_data['FH_m_h'] >= 122) & (full_data['FH_m_h'] <= 128)]
full_data['FH_m_h'] = full_data['FH_m_h'].abs()  # Take the absolute value of the Higgs mass
cuts = (full_data['FH_m_h']>=122) & (full_data['FH_m_h']<=128)

# full_data = full_data.copy()  # Ensure we have a copy of the full data for analysis
df_analysis = full_data[cuts].copy()

In [ ]:
# Physical masses — absolute values for plotting
df_analysis['m_chi_10'] = df_analysis['SP_m_chi_10'].abs()
df_analysis['m_chi_20'] = df_analysis['SP_m_chi_20'].abs()
df_analysis['Delta_m'] = df_analysis['m_chi_20'] - df_analysis['m_chi_10']

# Mass scales for parameter space plots — absolute values
df_analysis['M1_abs'] = df_analysis['SP_M_1'].abs()
df_analysis['M2_abs'] = df_analysis['SP_M_2'].abs()
df_analysis['mu_abs'] = df_analysis['SP_mu'].abs()

# Keep SP_M_1, SP_M_2, SP_mu as signed originals for fine-tuning

In [ ]:
df_analysis.info()

## Mass spectrum + $\Delta m$

In [ ]:
# Plot the distribution of $\Delta m$ for the filtered dataset

plt.figure(figsize=(10, 6))
sns.histplot(df_analysis['Delta_m'], bins=50, kde=True, color='orange', label='Surviving Models')
plt.axvline(x=91.1876, color='red', linestyle='--', linewidth=1.5, label=r'$m_Z$')
plt.axvline(x=125.10, color='green', linestyle='--', linewidth=1.5, label=r'$m_h$')
plt.legend()
plt.xlabel(r'$\Delta m$ (GeV)')
plt.ylabel('Number of Models')
# plt.title('Distribution of $\Delta m$ for Models with Higgs Mass Filter')
plt.grid(True)
plt.show()

In [ ]:
# Publication style settings
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'figure.dpi': 300,
    'text.usetex': True,  # remove if LaTeX not installed
})

fig, ax = plt.subplots(figsize=(8, 5))

# Histogram
sns.histplot(
    df_analysis['Delta_m'],
    bins=50,
    kde=True,
    color='steelblue',
    edgecolor='white',
    linewidth=0.4,
    alpha=0.8,
    line_kws={'linewidth': 1.5, 'color': 'navy'},
    ax=ax,
    label=r'Surviving models'
)

# Threshold lines
ax.axvline(
    x=91.1876,
    color='firebrick',
    linestyle='--',
    linewidth=1.5,
    label=r'$m_Z = 91.2$ GeV'
)
ax.axvline(
    x=125.10,
    color='forestgreen',
    linestyle='-.',
    linewidth=1.5,
    label=r'$m_h = 125.1$ GeV'
)

# Shade the intermediate regime M_Z < Delta m < m_h
# ax.axvspan(91.1876, 125.10, alpha=0.08, color='gold', label=r'$M_Z < \Delta m < m_h$ (Z-only regime)')

# Labels and formatting
ax.set_xlabel(r'$\Delta m = m_{\tilde{\chi}_2^0} - m_{\tilde{\chi}_1^0}$ (GeV)', labelpad=8)
ax.set_ylabel(r'Number of models', labelpad=8)
ax.legend(framealpha=0.9, edgecolor='gray', loc='upper right')
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
ax.set_xlim(left=0)
ax.yaxis.set_major_locator(ticker.AutoLocator())
ax.tick_marks_direction = 'in'
ax.tick_params(direction='in', top=True, right=True)

plt.tight_layout()
save_fig('delta_m_distribution.pdf')
plt.show()

In [ ]:
# Set a professional, clean aesthetic for scientific publishing
# sns.set_theme(style="ticks")
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "font.family": "serif"  # Matches standard LaTeX document fonts
})

plt.figure(figsize=(9, 5.5))  # Slightly tighter proportions look better in print

# Plot histogram with a professional edge color and slight transparency
sns.histplot(
    data=df_analysis, 
    x='Delta_m', 
    bins=60, 
    kde=True, 
    color='#1f77b4',       # Clean scientific blue
    edgecolor='white',     # Distinct separation between bins
    alpha=0.85, 
    label='Valid pMSSM Models'
)

# Reference lines for physical thresholds
plt.axvline(x=91.1876, color='#d62728', linestyle='--', linewidth=1.75, label=r'$m_Z$ threshold')
plt.axvline(x=125.09, color='#2ca02c', linestyle='-.', linewidth=1.75, label=r'$m_h$ threshold')

# Polish labels and layout
plt.xlabel(r'$\Delta m = m_{\tilde{\chi}_2^0} - m_{\tilde{\chi}_1^0}$ (GeV)')
plt.ylabel('Number of Models')
plt.xlim(0, df_analysis['Delta_m'].max() * 1.02)  # Adjust right boundary snugly to data limit

# Explicitly ensure the full frame box is kept on
ax = plt.gca()
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)

# Gridline decision: Explicitly off to avoid visual noise
plt.grid(False)

# Legend and layout cleanup
plt.legend(frameon=True, facecolor='white', edgecolor='none', loc='upper right')
plt.tight_layout()  # Keeps labels from clipping

plt.show()

In [ ]:
# Scatter plot of $m_{\tilde{\chi}_2^0}$ vs $m_{\tilde{\chi}_1^0}$ with $\Delta m = M_Z$ and $\Delta m = m_h$ lines overlaid

plt.figure(figsize=(10, 6))
plt.scatter(df_analysis['m_chi_10'], df_analysis['m_chi_20'], alpha=0.6, edgecolor='k', label='Surviving Models', marker='x')
plt.plot(df_analysis['m_chi_10'], df_analysis['m_chi_10'] + 91.1876, color='red', linestyle='--', linewidth=1.5, label=r'$\Delta m = m_Z$')
plt.plot(df_analysis['m_chi_10'], df_analysis['m_chi_10'] + 125.10, color='green', linestyle='--', linewidth=1.5, label=r'$\Delta m = m_h$')
plt.legend()
plt.xlabel(r'$m_{\tilde{\chi}_1^0}$ (GeV)')
plt.ylabel(r'$m_{\tilde{\chi}_2^0}$ (GeV)')
# plt.title('Scatter Plot of $m_{\tilde{\chi}_2^0}$ vs $m_{\tilde{\chi}_1^0}$ with $\Delta m$ Lines')
plt.grid(True)
plt.show()

In [ ]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'figure.dpi': 300,
    'text.usetex': True,
})

fig, ax = plt.subplots(figsize=(8, 6))  # taller than (8,5) to fix thinness

# Scatter plot
ax.scatter(
    df_analysis['m_chi_10'],
    df_analysis['m_chi_20'],
    color='steelblue',
    alpha=0.5,        # increase from 0.3
    s=4,              # increase from 1.5
    marker='.',
    rasterized=True,
    label=r'Surviving models',
    zorder=2
)

# Threshold lines
x_range = np.linspace(0, 2000, 1000)
ax.plot(
    x_range, x_range + 91.1876,
    color='firebrick',
    linestyle='--',
    linewidth=1.5,
    label=r'$\Delta m = m_Z \approx 91.2$ GeV',
    zorder=3
)
ax.plot(
    x_range, x_range + 125.10,
    color='forestgreen',
    linestyle='--',
    linewidth=1.5,
    label=r'$\Delta m = m_h \approx 125.1$ GeV',
    zorder=3
)

# Axes and formatting
ax.set_xlabel(r'$m_{\tilde{\chi}_1^0}$ (GeV)', labelpad=8)
ax.set_ylabel(r'$m_{\tilde{\chi}_2^0}$ (GeV)', labelpad=8)
ax.set_xlim(0, 2000)
ax.set_ylim(0, 2100)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

legend = ax.legend(
    framealpha=0.9,
    edgecolor='gray',
    # loc='upper left',
    markerscale=4,
    handlelength=2
)

plt.tight_layout()
save_fig('m_chi_scatter.pdf')
plt.show()

## Branching Ratios

In [ ]:
# print how many of the models are in the Z-only regime (M_Z < Delta m < m_h) and how many are in the Z and h regime (Delta m > m_h)

z_only_count = df_analysis[(df_analysis['Delta_m'] > 91.1876) & (df_analysis['Delta_m'] < 125.10)].shape[0]
z_and_h_count = df_analysis[df_analysis['Delta_m'] > df_analysis["FH_m_h"]].shape[0]

print(f'Number of models in Z-only regime (M_Z < Delta m < m_h): {z_only_count}')
print(f'Number of models in Z and h regime (Delta m > m_h): {z_and_h_count}')

z_only_percentage = (z_only_count / len(df_analysis)) * 100
print(f"Percentage of models in Z-only regime: {z_only_percentage:.2f}%")

z_and_h_percentage = (z_and_h_count / len(df_analysis)) * 100
print(f"Percentage of models in Z and h regime: {z_and_h_percentage:.2f}%")

In [ ]:
# Histogram of BR($\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z$)

plt.figure(figsize=(10, 6))
sns.histplot(
    df_analysis['SP_BF_chi_20_to_chi_10_Z'],
    bins=50,
    kde=True,
    label='Surviving Models',
    stat='count'
)
plt.xlabel(r'BR($\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z$)')
plt.ylabel('Number of Models')
# plt.title(r'Distribution of BR($\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z$) for Surviving Models')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.histplot(
    df_analysis['SP_BF_chi_20_to_chi_10_Z'],
    bins=50,
    kde=True,
    color='steelblue',
    edgecolor='white',
    linewidth=0.4,
    alpha=0.8,
    line_kws={'linewidth': 1.5, 'color': 'navy'},
    ax=ax,
    label=r'Surviving models',
    stat='count'
)

ax.set_xlabel(
    r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$',
    labelpad=8
)
ax.set_ylabel(r'Number of models', labelpad=8)
ax.set_xlim(0, 1)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
ax.legend(framealpha=0.9, edgecolor='gray', loc='upper right')

plt.tight_layout()
save_fig('br_z_distribution.pdf')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.histplot(
    df_analysis['SP_BF_chi_20_to_chi_10_h'],
    bins=50,
    kde=True,
    color='steelblue',
    edgecolor='white',
    linewidth=0.4,
    alpha=0.8,
    line_kws={'linewidth': 1.5, 'color': 'navy'},
    ax=ax,
    label=r'Surviving models',
    stat='count'
)

ax.set_xlabel(
    r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$',
    labelpad=8
)
ax.set_ylabel(r'Number of models', labelpad=8)
ax.set_xlim(0, 1)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
ax.legend(framealpha=0.9, edgecolor='gray', loc='upper right')

plt.tight_layout()
save_fig('br_h_distribution.pdf')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# --- Colours ---
color_z = 'steelblue'
color_h = 'firebrick'
color_other = 'forestgreen'

# --- Define quantities ---
Delta_m = df_analysis['Delta_m']
br_z = df_analysis['SP_BF_chi_20_to_chi_10_Z']
br_h = df_analysis['SP_BF_chi_20_to_chi_10_h']
br_other = 1 - br_z - br_h  # everything else

# --- Bins ---
bins = np.linspace(0, Delta_m.max(), 51)
bin_centres = 0.5 * (bins[:-1] + bins[1:])

# --- Bin statistics ---
def bin_stats(values, Delta_m, bins):
    medians, lows, highs = [], [], []
    for i in range(len(bins) - 1):
        mask = (Delta_m >= bins[i]) & (Delta_m < bins[i+1])
        vals = values[mask]
        if len(vals) > 5:  # minimum models per bin
            medians.append(np.median(vals))
            lows.append(np.percentile(vals, 16))
            highs.append(np.percentile(vals, 84))
        else:
            medians.append(np.nan)
            lows.append(np.nan)
            highs.append(np.nan)
    return np.array(medians), np.array(lows), np.array(highs)

z_med, z_low, z_high = bin_stats(br_z, Delta_m, bins)
h_med, h_low, h_high = bin_stats(br_h, Delta_m, bins)
o_med, o_low, o_high = bin_stats(br_other, Delta_m, bins)

# --- Z channel ---
ax.plot(bin_centres, z_med,
        color=color_z, linewidth=1.5,
        label=r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$')
ax.fill_between(bin_centres, z_low, z_high,
                color=color_z, alpha=0.15)

# --- h channel ---
ax.plot(bin_centres, h_med,
        color=color_h, linewidth=1.5,
        label=r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$')
ax.fill_between(bin_centres, h_low, h_high,
                color=color_h, alpha=0.15)

# --- Other decays ---
ax.plot(bin_centres, o_med,
        color=color_other, linewidth=1.5, linestyle='--',
        label=r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \mathrm{other})$')
# No 1-sigma band for other — cleaner plot, supervisor can advise

# --- Threshold lines ---
ax.axvline(91.1876, color='black', linestyle='--',
           linewidth=1.5, label=r'$m_Z \approx 91.2$ GeV')
ax.axvline(125.25, color='gray', linestyle='-.',
           linewidth=1.5, label=r'$m_h \approx 125.3$ GeV')

# --- Formatting ---
ax.set_xlabel(r'$\Delta m$ (GeV)', labelpad=8)
ax.set_ylabel(r'Branching Ratio', labelpad=8)
ax.set_xlim(0, Delta_m.max())
ax.set_ylim(0, 1.05)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
ax.legend(framealpha=0.9, edgecolor='gray', loc='upper right')

plt.tight_layout()
# save_fig('br_evolution.pdf', bbox_inches='tight')
plt.show()

In [ ]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'figure.dpi': 300,
    'text.usetex': True,
})

fig, ax = plt.subplots(figsize=(8, 5))

# --- Colours ---
color_z = 'steelblue'
color_h = 'firebrick'
color_other = 'forestgreen'

# --- Define quantities ---
Delta_m = df_analysis['Delta_m']
br_z = df_analysis['SP_BF_chi_20_to_chi_10_Z']
br_h = df_analysis['SP_BF_chi_20_to_chi_10_h']
br_other = 1 - br_z - br_h

# --- Fine bins for Z and h ---
bins_fine = np.linspace(0, Delta_m.max(), 51)
bin_centres_fine = 0.5 * (bins_fine[:-1] + bins_fine[1:])

# --- Coarse bins for other ---
bins_coarse = np.linspace(0, Delta_m.max(), 26)
bin_centres_coarse = 0.5 * (bins_coarse[:-1] + bins_coarse[1:])

# --- Bin statistics function ---
def bin_stats(values, Delta_m, bins, min_count=5):
    medians, lows, highs = [], [], []
    for i in range(len(bins) - 1):
        mask = (Delta_m >= bins[i]) & (Delta_m < bins[i+1])
        vals = values[mask]
        if len(vals) > min_count:
            medians.append(np.median(vals))
            lows.append(np.percentile(vals, 16))
            highs.append(np.percentile(vals, 84))
        else:
            medians.append(np.nan)
            lows.append(np.nan)
            highs.append(np.nan)
    return np.array(medians), np.array(lows), np.array(highs)

# --- Compute statistics ---
z_med, z_low, z_high = bin_stats(br_z, Delta_m, bins_fine)
h_med, h_low, h_high = bin_stats(br_h, Delta_m, bins_fine)
o_med, _, _ = bin_stats(br_other, Delta_m, bins_coarse, min_count=10)

# --- Z channel ---
ax.plot(bin_centres_fine, z_med,
        color=color_z, linewidth=1.5,
        label=r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$',
        zorder=3)
ax.fill_between(bin_centres_fine, z_low, z_high,
                color=color_z, alpha=0.15, zorder=2)

# --- h channel ---
ax.plot(bin_centres_fine, h_med,
        color=color_h, linewidth=1.5,
        label=r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$',
        zorder=3)
ax.fill_between(bin_centres_fine, h_low, h_high,
                color=color_h, alpha=0.15, zorder=2)

# --- Other decays ---
ax.plot(bin_centres_coarse, o_med,
        color=color_other, linewidth=1.5, linestyle='--',
        label=r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \mathrm{other})$',
        zorder=3)

# --- Threshold lines ---
ax.axvline(91.1876, color='black', linestyle='--',
           linewidth=1.5, label=r'$m_Z \approx 91.2$ GeV', zorder=4)
ax.axvline(125.25, color='gray', linestyle='-.',
           linewidth=1.5, label=r'$m_h \approx 125.3$ GeV', zorder=4)

# --- Formatting ---
ax.set_xlabel(r'$\Delta m$ (GeV)', labelpad=8)
ax.set_ylabel(r'Branching Ratio', labelpad=8)
ax.set_xlim(0, Delta_m.max())
ax.set_ylim(0, 1.05)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
ax.legend(framealpha=0.9, edgecolor='gray', loc='upper right')

plt.tight_layout()
save_fig('br_evolution.pdf')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

sc = ax.scatter(
    df_analysis['SP_BF_chi_20_to_chi_10_Z'],
    df_analysis['SP_BF_chi_20_to_chi_10_h'],
    c=df_analysis['Delta_m'],
    cmap='viridis',
    s=1.5,
    alpha=0.4,
    rasterized=True
)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label(r'$\Delta m$ (GeV)', labelpad=8)

# Diagonal line where BR(Z) = BR(h)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1,
        label=r'$\mathrm{BR}(Z) = \mathrm{BR}(h)$', alpha=0.5)

ax.set_xlabel(
    r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$',
    labelpad=8)
ax.set_ylabel(
    r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$',
    labelpad=8)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
ax.legend(framealpha=0.9, edgecolor='gray')

plt.tight_layout()
save_fig('br_z_vs_h.pdf')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Filter to on-shell regime only
df_onshell = df_analysis[df_analysis['Delta_m'] > 125.10]

for ax, col, label in zip(
    [ax1, ax2],
    ['SP_BF_chi_20_to_chi_10_Z', 'SP_BF_chi_20_to_chi_10_h'],
    [r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$',
     r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$']
):
    sns.histplot(
        df_onshell[col],
        bins=50,
        kde=True,
        color='steelblue',
        edgecolor='white',
        linewidth=0.4,
        alpha=0.8,
        line_kws={'linewidth': 1.5, 'color': 'navy'},
        ax=ax,
        stat='count'
    )
    ax.set_xlabel(label, labelpad=8)
    ax.set_ylabel(r'Number of models', labelpad=8)
    ax.set_xlim(0, 1)
    ax.tick_params(direction='in', top=True, right=True)
    ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

plt.tight_layout()
save_fig('br_histograms.pdf')
plt.show()

In [ ]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.dpi': 300,
    'text.usetex': True,
})

mZ = 91.1876
mh = 125.25

Delta_m = df_analysis['Delta_m']
br_z = df_analysis['SP_BF_chi_20_to_chi_10_Z']

masks = [
    Delta_m < mZ,
    (Delta_m >= mZ) & (Delta_m < mh),
    (Delta_m >= mh) & (Delta_m < 250.0),
    (Delta_m >= 250.0) & (Delta_m < 500.0),
    (Delta_m >= 500.0) & (Delta_m < 750.0),
    Delta_m >= 750.0
]

titles = [
    r'Off-shell regime ($\Delta m < m_Z$)',
    r'$Z$-only regime ($m_Z \leq \Delta m < m_h$)',
    r'Both channels open ($m_h \leq \Delta m < 250$ GeV)',
    r'High-$\Delta m$ regime 1 ($250 \leq \Delta m < 500$ GeV)',
    r'High-$\Delta m$ regime 2 ($500 \leq \Delta m < 750$ GeV)',
    r'Asymptotic limit ($\Delta m \geq 750$ GeV)'
]

colors = [
    'steelblue',
    'firebrick',
    'forestgreen',
    'darkorange',
    'mediumpurple',
    'saddlebrown'
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey=False)
axes_flat = axes.flatten()

for i, mask in enumerate(masks):
    ax = axes_flat[i]
    data_slice = br_z[mask]

    if len(data_slice) > 0:
        sns.histplot(
            data_slice,
            bins=40,
            kde=True,
            ax=ax,
            color=colors[i],
            edgecolor='white',
            linewidth=0.4,
            alpha=0.75,
            stat='density',
            line_kws={'linewidth': 1.5, 'color': colors[i]}
        )

        p16 = np.percentile(data_slice, 16)
        p50 = np.median(data_slice)
        p84 = np.percentile(data_slice, 84)
        n = mask.sum()

        ax.text(
            0.95, 0.92,
            f'$N = {n}$\n'
            f'16th: {p16:.2f}\n'
            f'Median: {p50:.2f}\n'
            f'84th: {p84:.2f}',
            transform=ax.transAxes,
            ha='right', va='top',
            fontsize=9,
            bbox=dict(boxstyle='round,pad=0.4', fc='white',
                      ec='gray', alpha=0.9, lw=0.8)
        )
    else:
        ax.text(0.5, 0.5, 'No data', transform=ax.transAxes,
                ha='center', va='center')

    ax.set_title(titles[i], pad=6)
    ax.set_xlabel(
        r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$',
        labelpad=6
    )
    ax.set_ylabel(r'Probability density', labelpad=6)
    ax.set_xlim(0, 1)
    ax.tick_params(direction='in', top=True, right=True)
    ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

plt.tight_layout()
save_fig('br_z_regimes.pdf')
plt.show()

In [ ]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.dpi': 300,
    'text.usetex': True,
})

mZ = 91.1876
mh = 125.25

Delta_m = df_analysis['Delta_m']
br_h = df_analysis['SP_BF_chi_20_to_chi_10_h']

masks = [
    Delta_m < mZ,
    (Delta_m >= mZ) & (Delta_m < mh),
    (Delta_m >= mh) & (Delta_m < 250.0),
    (Delta_m >= 250.0) & (Delta_m < 500.0),
    (Delta_m >= 500.0) & (Delta_m < 750.0),
    Delta_m >= 750.0
]

titles = [
    r'Off-shell regime ($\Delta m < m_Z$)',
    r'$Z$-only regime ($m_Z \leq \Delta m < m_h$)',
    r'Both channels open ($m_h \leq \Delta m < 250$ GeV)',
    r'High-$\Delta m$ regime 1 ($250 \leq \Delta m < 500$ GeV)',
    r'High-$\Delta m$ regime 2 ($500 \leq \Delta m < 750$ GeV)',
    r'Asymptotic limit ($\Delta m \geq 750$ GeV)'
]

colors = [
    'steelblue',
    'firebrick',
    'forestgreen',
    'darkorange',
    'mediumpurple',
    'saddlebrown'
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey=False)
axes_flat = axes.flatten()

for i, mask in enumerate(masks):
    ax = axes_flat[i]
    data_slice = br_h[mask]

    if len(data_slice) > 0:
        sns.histplot(
            data_slice,
            bins=40,
            kde=True,
            ax=ax,
            color=colors[i],
            edgecolor='white',
            linewidth=0.4,
            alpha=0.75,
            stat='density',
            line_kws={'linewidth': 1.5, 'color': colors[i]}
        )

        p16 = np.percentile(data_slice, 16)
        p50 = np.median(data_slice)
        p84 = np.percentile(data_slice, 84)
        n = mask.sum()

        ax.text(
            0.95, 0.92,
            f'$N = {n}$\n'
            f'16th: {p16:.2f}\n'
            f'Median: {p50:.2f}\n'
            f'84th: {p84:.2f}',
            transform=ax.transAxes,
            ha='right', va='top',
            fontsize=9,
            bbox=dict(boxstyle='round,pad=0.4', fc='white',
                      ec='gray', alpha=0.9, lw=0.8)
        )
    else:
        ax.text(0.5, 0.5, 'No data', transform=ax.transAxes,
                ha='center', va='center')

    ax.set_title(titles[i], pad=6)
    ax.set_xlabel(
        r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$',
        labelpad=6
    )
    ax.set_ylabel(r'Probability density', labelpad=6)
    ax.set_xlim(0, 1)
    ax.tick_params(direction='in', top=True, right=True)
    ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

plt.tight_layout()
save_fig('br_h_regimes.pdf')
plt.show()

## Correlations with Neutralino Mixing

In [ ]:
# Scatter: BR(Z) vs higgsino fraction of $\tilde{\chi}_1^0$

plt.figure(figsize=(10, 6))
plt.scatter(
    df_analysis['SP_BF_chi_20_to_chi_10_Z'],
    df_analysis['SP_chi_10_Higgsino_frac'],
    alpha=0.6,
    edgecolor='k',
    label='Surviving Models',
    marker='x'
)
plt.xlabel(r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$')
plt.ylabel(r'Higgsino fraction of $\tilde{\chi}_1^0$')
# plt.title('Scatter Plot of BR(Z) vs Higgsino Fraction of $\tilde{\chi}_1^0$')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: BR(Z) vs higgsino fraction of $\tilde{\chi}_2^0$

plt.figure(figsize=(10, 6))
plt.scatter(
    df_analysis['SP_BF_chi_20_to_chi_10_Z'],
    df_analysis['SP_chi_20_Higgsino_frac'],
    alpha=0.6,
    edgecolor='k',
    label='Surviving Models',
    marker='x'
)
plt.xlabel(r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$')
plt.ylabel(r'Higgsino fraction of $\tilde{\chi}_2^0$')
# plt.title('Scatter Plot of BR(Z) vs Higgsino Fraction of $\tilde{\chi}_2^0$')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: BR(h) vs higgsino fraction of $\tilde{\chi}_1^0$

plt.figure(figsize=(10, 6))
plt.scatter(
    df_analysis['SP_BF_chi_20_to_chi_10_h'],
    df_analysis['SP_chi_10_Higgsino_frac'],
    alpha=0.6,
    edgecolor='k',
    label='Surviving Models',
    marker='x'
)
plt.xlabel(r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$')
plt.ylabel(r'Higgsino fraction of $\tilde{\chi}_1^0$')
# plt.title('Scatter Plot of BR(h) vs Higgsino Fraction of $\tilde{\chi}_1^0$')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: BR(h) vs higgsino fraction of $\tilde{\chi}_2^0$

plt.figure(figsize=(10, 6))
plt.scatter(
    df_analysis['SP_BF_chi_20_to_chi_10_h'],
    df_analysis['SP_chi_20_Higgsino_frac'],
    alpha=0.6,
    edgecolor='k',
    label='Surviving Models',
    marker='x'
)
plt.xlabel(r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$')
plt.ylabel(r'Higgsino fraction of $\tilde{\chi}_2^0$')
# plt.title('Scatter Plot of BR(h) vs Higgsino Fraction of $\tilde{\chi}_2^0$')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 2D density plot: BR(Z) in the $M_1$ vs $\mu$ plane

plt.figure(figsize=(10, 6))
sc = plt.scatter(
    df_analysis['SP_M_1'],
    df_analysis['SP_mu'],
    c=df_analysis['SP_BF_chi_20_to_chi_10_Z'],
    cmap='viridis',
    s=1.5,
    alpha=0.4,
    rasterized=True
)
cbar = plt.colorbar(sc)
cbar.set_label(r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$')
plt.xlabel(r'$M_1$ (GeV)')
plt.ylabel(r'$\mu$ (GeV)')
# plt.title('BR(Z) in the $M_1$ vs $\mu$ plane')
plt.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# 2D density plot: BR(Z) in the $M_2$ vs $\mu$ plane

plt.figure(figsize=(10, 6))
sc = plt.scatter(
    df_analysis['SP_M_2'],
    df_analysis['SP_mu'],
    c=df_analysis['SP_BF_chi_20_to_chi_10_Z'],
    cmap='viridis',
    s=1.5,
    alpha=0.4,
    rasterized=True
)
cbar = plt.colorbar(sc)
cbar.set_label(r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$')
plt.xlabel(r'$M_2$ (GeV)')
plt.ylabel(r'$\mu$ (GeV)')
# plt.title('BR(Z) in the $M_2$ vs $\mu$ plane')
plt.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# 2D density plot: BR(h) in the $M_1$ vs $\mu$ plane

plt.figure(figsize=(10, 6))
sc = plt.scatter(
    df_analysis['SP_M_1'],
    df_analysis['SP_mu'],
    c=df_analysis['SP_BF_chi_20_to_chi_10_h'],
    cmap='viridis',
    s=1.5,
    alpha=0.4,
    rasterized=True
)
cbar = plt.colorbar(sc)
cbar.set_label(r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$')
plt.xlabel(r'$M_1$ (GeV)')
plt.ylabel(r'$\mu$ (GeV)')
# plt.title('BR(h) in the $M_1$ vs $\mu$ plane')
plt.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# 2D density plot: BR(h) in the $M_2$ vs $\mu$ plane

plt.figure(figsize=(10, 6))
sc = plt.scatter(
    df_analysis['SP_M_2'],
    df_analysis['SP_mu'],
    c=df_analysis['SP_BF_chi_20_to_chi_10_h'],
    cmap='viridis',
    s=1.5,
    alpha=0.4,
    rasterized=True
)
cbar = plt.colorbar(sc)
cbar.set_label(r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$')
plt.xlabel(r'$M_2$ (GeV)')
plt.ylabel(r'$\mu$ (GeV)')
# plt.title('BR(h) in the $M_2$ vs $\mu$ plane')
plt.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# 2D density plot: BR(Z) in the $M_1$ vs $M_2$ plane

plt.figure(figsize=(10, 6))
sc = plt.scatter(
    df_analysis['SP_M_1'],
    df_analysis['SP_M_2'],
    c=df_analysis['SP_BF_chi_20_to_chi_10_Z'],
    cmap='viridis',
    s=1.5,
    alpha=0.4,
    rasterized=True
)
cbar = plt.colorbar(sc)
cbar.set_label(r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$')
plt.xlabel(r'$M_1$ (GeV)')
plt.ylabel(r'$M_2$ (GeV)')
# plt.title('BR(Z) in the $M_1$ vs $M_2$ plane')
plt.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# 2D density plot: BR(h) in the $M_1$ vs $M_2$ plane

plt.figure(figsize=(10, 6))
sc = plt.scatter(
    df_analysis['SP_M_1'],
    df_analysis['SP_M_2'],
    c=df_analysis['SP_BF_chi_20_to_chi_10_h'],
    cmap='viridis',
    s=1.5,
    alpha=0.4,
    rasterized=True
)
cbar = plt.colorbar(sc)
cbar.set_label(r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$')
plt.xlabel(r'$M_1$ (GeV)')
plt.ylabel(r'$M_2$ (GeV)')
# plt.title('BR(h) in the $M_1$ vs $M_2$ plane')
plt.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
df_analysis["D12"] = df_analysis['SP_N_23'] * df_analysis['SP_N_13'] - df_analysis['SP_N_24'] * df_analysis['SP_N_14']

df_analysis = df_analysis.copy()

In [ ]:
df_analysis.info()

In [ ]:
# ── 0. Derived quantities ────────────────────────────────────────────────────

MZ = 91.1876
mh = 125.1

df_analysis["D12_sq"]  = df_analysis["D12"] ** 2

# Higgsino fraction of chi_20 (sum of squared higgsino components)
df_analysis["higgs_frac_chi20"] = (
    df_analysis["SP_N_23"] ** 2 + df_analysis["SP_N_24"] ** 2
)

df_analysis = df_analysis.copy()  # defragment

# ── 1. Filter to on-shell regime ─────────────────────────────────────────────

df_onshell = df_analysis[df_analysis["Delta_m"] > MZ].copy()

# Subsets for the two kinematic regimes
df_Zonly  = df_onshell[df_onshell["Delta_m"] < mh]   # Z-only: MZ < Δm < mh
df_both   = df_onshell[df_onshell["Delta_m"] >= mh]   # Both channels open

BR_Z = "SP_BF_chi_20_to_chi_10_Z"
BR_h = "SP_BF_chi_20_to_chi_10_h"

# ── 2. Plot 1: BR(Z) vs |D12|² ───────────────────────────────────────────────
# Direct analytic validation of Eq. 3.19

fig, ax = plt.subplots(figsize=(7, 5))

sc = ax.scatter(
    df_both["D12_sq"],
    df_both[BR_Z],
    c=df_both["Delta_m"],
    cmap="viridis",
    s=3,
    alpha=0.5,
    rasterized=True,
)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(r"$\Delta m$ (GeV)", fontsize=12)

ax.set_xlabel(r"$|\mathcal{D}_{12}|^2 = (N_{23}N_{13} - N_{24}N_{14})^2$", fontsize=12)
ax.set_ylabel(r"$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$", fontsize=12)
# ax.set_title(r"Analytic coupling factor vs $Z$-channel branching ratio", fontsize=13)
ax.set_xlim(0, None)
ax.set_ylim(0, 1)

plt.tight_layout()
# save_fig('BR_Z_vs_D12sq.pdf', bbox_inches='tight')
plt.show()

# ── 3. Plot 2: BR(Z) and BR(h) vs higgsino fraction of χ̃⁰₂ ─────────────────
# colour-coded by χ̃⁰₁ higgsino fraction to capture the D12 cancellation

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

for ax, br_col, label in zip(
    axes,
    [BR_Z, BR_h],
    [r"$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$",
     r"$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$"],
):
    sc = ax.scatter(
        df_both["higgs_frac_chi20"],
        df_both[br_col],
        c=df_both["SP_chi_10_Higgsino_frac"],
        # cmap="plasma",
        cmap="viridis",
        s=3,
        alpha=0.5,
        rasterized=True,
    )
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(r"Higgsino fraction of $\tilde{\chi}_1^0$", fontsize=11)

    ax.set_xlabel(
        r"Higgsino fraction of $\tilde{\chi}_2^0$"
        r"$\ (|N_{23}|^2 + |N_{24}|^2)$",
        fontsize=11,
    )
    ax.set_ylabel(label, fontsize=12)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.suptitle(
    r"Branching ratios vs $\tilde{\chi}_2^0$ higgsino fraction,"
    r" coloured by $\tilde{\chi}_1^0$ higgsino fraction",
    fontsize=13,
)
plt.tight_layout()
# save_fig("BR_vs_higgsino_frac.pdf", bbox_inches='tight')
plt.show()

# ── 4. Plot 3: 2D density — BR(Z) in M1 vs µ plane ──────────────────────────

fig, ax = plt.subplots(figsize=(7, 6))

sc = ax.scatter(
    df_both["M1_abs"],
    df_both["mu_abs"],
    c=df_both[BR_Z],
    cmap="RdYlBu",
    s=4,
    alpha=0.6,
    vmin=0,
    vmax=1,
    rasterized=True,
)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(r"$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$", fontsize=12)

ax.set_xlabel(r"$|M_1|$ (GeV)", fontsize=12)
ax.set_ylabel(r"$|\mu|$ (GeV)", fontsize=12)
ax.set_title(
    r"$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$ in the $M_1$–$\mu$ plane",
    fontsize=13,
)

plt.tight_layout()
# save_fig("BR_Z_M1_mu_plane.pdf", bbox_inches='tight')
plt.show()

In [ ]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'figure.dpi': 300,
    'text.usetex': True,
})

# --- Derived quantities ---
df_analysis['D12'] = (
    df_analysis['SP_N_23'] * df_analysis['SP_N_13']
    - df_analysis['SP_N_24'] * df_analysis['SP_N_14']
)
df_analysis['D12_sq'] = df_analysis['D12'] ** 2

# --- Filter to both channels open ---
df_both = df_analysis[df_analysis['Delta_m'] >= 125.25].copy()

BR_Z = 'SP_BF_chi_20_to_chi_10_Z'

# --- Plot ---
fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(
    df_both['D12_sq'],
    df_both[BR_Z],
    c=df_both['Delta_m'],
    cmap='viridis',
    s=1.5,
    alpha=0.4,
    rasterized=True,
)

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(r'$\Delta m$ (GeV)', labelpad=8)

ax.set_xlabel(
    r'$|\mathcal{D}_{12}|^2 = (N_{23}N_{13} - N_{24}N_{14})^2$',
    labelpad=8
)
ax.set_ylabel(
    r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$',
    labelpad=8
)
ax.set_xlim(0, None)
ax.set_ylim(0, 1)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

plt.tight_layout()
# save_fig('BR_Z_vs_D12sq.pdf', bbox_inches='tight')
plt.show()

In [ ]:
alpha = df_analysis['alpha']

df_analysis['C12'] = (
    (df_analysis['SP_N_22'] - df_analysis['tan_tW'] * df_analysis['SP_N_21'])
    * (np.sin(alpha) * df_analysis['SP_N_13'] + np.cos(alpha) * df_analysis['SP_N_14'])
    + (df_analysis['SP_N_12'] - df_analysis['tan_tW'] * df_analysis['SP_N_11'])
    * (np.sin(alpha) * df_analysis['SP_N_23'] + np.cos(alpha) * df_analysis['SP_N_24'])
)
df_analysis['C12_sq'] = df_analysis['C12'] ** 2

# --- Filter ---
df_both = df_analysis[df_analysis['Delta_m'] >= 125.25].copy()

BR_h = 'SP_BF_chi_20_to_chi_10_h'

# --- Plot ---
fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(
    df_both['C12_sq'],
    df_both[BR_h],
    c=df_both['Delta_m'],
    cmap='viridis',
    s=1.5,
    alpha=0.4,
    rasterized=True,
)

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(r'$\Delta m$ (GeV)', labelpad=8)

ax.set_xlabel(
    r'$|\mathcal{C}_{12}|^2$',
    labelpad=8
)
ax.set_ylabel(
    r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$',
    labelpad=8
)
ax.set_xlim(0, None)
ax.set_ylim(0, 1)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

plt.tight_layout()
# save_fig('BR_h_vs_C12sq.pdf', bbox_inches='tight')
plt.show()

In [ ]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'figure.dpi': 300,
    'text.usetex': True,
})

# --- Derived quantities ---
m_Z = 91.1876
df_analysis['tan_tW'] = np.sqrt(
    (m_Z**2 - df_analysis['SP_m_W']**2) / df_analysis['SP_m_W']**2
)

df_analysis['D12'] = (
    df_analysis['SP_N_23'] * df_analysis['SP_N_13']
    - df_analysis['SP_N_24'] * df_analysis['SP_N_14']
)
df_analysis['D12_sq'] = df_analysis['D12'] ** 2

alpha = df_analysis['alpha']
df_analysis['C12'] = (
    (df_analysis['SP_N_22'] - df_analysis['tan_tW'] * df_analysis['SP_N_21'])
    * (np.sin(alpha) * df_analysis['SP_N_13'] + np.cos(alpha) * df_analysis['SP_N_14'])
    + (df_analysis['SP_N_12'] - df_analysis['tan_tW'] * df_analysis['SP_N_11'])
    * (np.sin(alpha) * df_analysis['SP_N_23'] + np.cos(alpha) * df_analysis['SP_N_24'])
)
df_analysis['C12_sq'] = df_analysis['C12'] ** 2

# --- Filter ---
df_both = df_analysis[df_analysis['Delta_m'] >= 125.25].copy()

BR_Z = 'SP_BF_chi_20_to_chi_10_Z'
BR_h = 'SP_BF_chi_20_to_chi_10_h'

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: BR(Z) vs D12^2
sc0 = axes[0].scatter(
    df_both['D12_sq'],
    df_both[BR_Z],
    c=df_both['Delta_m'],
    cmap='viridis',
    s=1.5,
    alpha=0.4,
    rasterized=True,
)
cbar0 = fig.colorbar(sc0, ax=axes[0])
cbar0.set_label(r'$\Delta m$ (GeV)', labelpad=8)
axes[0].set_xlabel(
    r'$|\mathcal{D}_{12}|^2$',
    labelpad=8
)
axes[0].set_ylabel(
    r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$',
    labelpad=8
)
axes[0].set_xlim(0, None)
axes[0].set_ylim(0, 1)
axes[0].tick_params(direction='in', top=True, right=True)
axes[0].grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

# Right: BR(h) vs C12^2
sc1 = axes[1].scatter(
    df_both['C12_sq'],
    df_both[BR_h],
    c=df_both['Delta_m'],
    cmap='viridis',
    s=1.5,
    alpha=0.4,
    rasterized=True,
)
cbar1 = fig.colorbar(sc1, ax=axes[1])
cbar1.set_label(r'$\Delta m$ (GeV)', labelpad=8)
axes[1].set_xlabel(
    r'$|\mathcal{C}_{12}|^2$',
    labelpad=8
)
axes[1].set_ylabel(
    r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$',
    labelpad=8
)
axes[1].set_xlim(0, None)
axes[1].set_ylim(0, 1)
axes[1].tick_params(direction='in', top=True, right=True)
axes[1].grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

plt.tight_layout()
save_fig('BR_vs_couplings.pdf')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)

BR_Z = 'SP_BF_chi_20_to_chi_10_Z'
BR_h = 'SP_BF_chi_20_to_chi_10_h'

for ax, br_col, ylabel in zip(
    axes,
    [BR_Z, BR_h],
    [r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$',
     r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$'],
):
    sc = ax.scatter(
        df_both['SP_chi_20_Higgsino_frac'],
        df_both[br_col],
        c=df_both['SP_chi_10_Higgsino_frac'],
        # cmap='plasma',
        cmap='viridis',
        s=1.5,
        alpha=0.4,
        rasterized=True,
    )
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(
        r'Higgsino fraction of $\tilde{\chi}_1^0$',
        labelpad=8
    )
    ax.set_xlabel(
        r'Higgsino fraction of $\tilde{\chi}_2^0$',
        labelpad=8
    )
    ax.set_ylabel(ylabel, labelpad=8)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.tick_params(direction='in', top=True, right=True)
    ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

plt.tight_layout()
save_fig('BR_vs_higgsino_frac.pdf')
plt.show()

In [ ]:
BR_Z = 'SP_BF_chi_20_to_chi_10_Z'
BR_h = 'SP_BF_chi_20_to_chi_10_h'

eps = 1e-6

# Vectorised — no loop
df_ratio = df_both.copy()
df_ratio = df_ratio[(df_ratio[BR_Z] > 0) | (df_ratio[BR_h] > 0)]
df_ratio[BR_Z] = df_ratio[BR_Z].clip(lower=eps)
df_ratio[BR_h] = df_ratio[BR_h].clip(lower=eps)
df_ratio['BR_ratio'] = df_ratio[BR_Z] / df_ratio[BR_h]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, frac_col, xlabel in zip(
    axes,
    ['SP_chi_20_Higgsino_frac', 'SP_chi_20_Bino_frac'],
    [r'Higgsino fraction of $\tilde{\chi}_2^0$',
     r'Bino fraction of $\tilde{\chi}_2^0$'],
):
    sc = ax.scatter(
        df_ratio[frac_col],
        df_ratio['BR_ratio'],
        c=df_ratio['Delta_m'],
        cmap='viridis',
        s=1.5,
        alpha=0.4,
        rasterized=True,
    )
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(r'$\Delta m$ (GeV)', labelpad=8)
    ax.set_xlabel(xlabel, labelpad=8)
    ax.set_ylabel(
        r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$'
        r'$/\,\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$',
        labelpad=8
    )
    ax.set_yscale('log')
    ax.set_xlim(0, 1)
    ax.tick_params(direction='in', top=True, right=True)
    ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

plt.tight_layout()
# save_fig('BR_ratio_vs_composition.pdf', bbox_inches='tight')
plt.show()

In [ ]:
BR_Z = 'SP_BF_chi_20_to_chi_10_Z'
BR_h = 'SP_BF_chi_20_to_chi_10_h'

eps = 1e-6

# Vectorised — no loop
df_ratio = df_both.copy()
df_ratio = df_ratio[(df_ratio[BR_Z] > 0) | (df_ratio[BR_h] > 0)]
df_ratio[BR_Z] = df_ratio[BR_Z].clip(lower=eps)
df_ratio[BR_h] = df_ratio[BR_h].clip(lower=eps)
df_ratio['BR_ratio'] = df_ratio[BR_Z] / df_ratio[BR_h]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, frac_col, xlabel in zip(
    axes,
    ['SP_chi_20_Higgsino_frac',
     'SP_chi_20_Bino_frac',
     'SP_chi_20_Wino_frac'],
    [r'Higgsino fraction of $\tilde{\chi}_2^0$',
     r'Bino fraction of $\tilde{\chi}_2^0$',
     r'Wino fraction of $\tilde{\chi}_2^0$'],
):
    sc = ax.scatter(
        df_ratio[frac_col],
        df_ratio['BR_ratio'],
        c=df_ratio['Delta_m'],
        cmap='viridis',
        s=1.5,
        alpha=0.4,
        rasterized=True,
    )
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(r'$\Delta m$ (GeV)', labelpad=8)
    ax.set_xlabel(xlabel, labelpad=8)
    ax.set_ylabel(
        r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$'
        r'$/\,\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$',
        labelpad=8
    )
    ax.set_yscale('log')
    ax.set_xlim(0, 1)
    ax.tick_params(direction='in', top=True, right=True)
    ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

plt.tight_layout()
save_fig('BR_ratio_vs_composition.pdf')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(
    df_both['M1_abs'],
    df_both['mu_abs'],
    c=df_both['SP_BF_chi_20_to_chi_10_Z'],
    cmap='RdYlBu',
    s=1.5,
    alpha=0.6,
    vmin=0,
    vmax=1,
    rasterized=True,
)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(
    r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$',
    labelpad=8
)
ax.set_xlabel(r'$|M_1|$ (GeV)', labelpad=8)
ax.set_ylabel(r'$|\mu|$ (GeV)', labelpad=8)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

plt.tight_layout()
# save_fig('BR_Z_M1_mu_plane.pdf', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(
    df_both['M1_abs'],
    df_both['mu_abs'],
    c=df_both['SP_BF_chi_20_to_chi_10_Z'],
    cmap='viridis',
    s=1.5,
    alpha=0.6,
    vmin=0,
    vmax=1,
    rasterized=True,
)

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(
    r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$',
    labelpad=8
)

ax.set_xlabel(r'$|M_1|$ (GeV)', labelpad=8)
ax.set_ylabel(r'$|\mu|$ (GeV)', labelpad=8)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

plt.tight_layout()
save_fig('BR_Z_M1_mu_plane.pdf')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(
    df_both['M2_abs'],
    df_both['mu_abs'],
    c=df_both['SP_BF_chi_20_to_chi_10_Z'],
    cmap='viridis',
    s=1.5,
    alpha=0.6,
    vmin=0,
    vmax=1,
    rasterized=True,
)

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(
    r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$',
    labelpad=8
)

ax.set_xlabel(r'$|M_2|$ (GeV)', labelpad=8)
ax.set_ylabel(r'$|\mu|$ (GeV)', labelpad=8)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

plt.tight_layout()
save_fig('BR_Z_M2_mu_plane.pdf')
plt.show()

In [ ]:
BR_Z = 'SP_BF_chi_20_to_chi_10_Z'
BR_h = 'SP_BF_chi_20_to_chi_10_h'

plots = [
    (BR_Z, 'M1_abs', 'mu_abs',
     r'$|M_1|$ (GeV)', r'$|\mu|$ (GeV)',
     r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$'),
    (BR_Z, 'M2_abs', 'mu_abs',
     r'$|M_2|$ (GeV)', r'$|\mu|$ (GeV)',
     r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$'),
    (BR_h, 'M1_abs', 'mu_abs',
     r'$|M_1|$ (GeV)', r'$|\mu|$ (GeV)',
     r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$'),
    (BR_h, 'M2_abs', 'mu_abs',
     r'$|M_2|$ (GeV)', r'$|\mu|$ (GeV)',
     r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for ax, (br_col, x_col, y_col, xlabel, ylabel, cbarlabel) in zip(
    axes.flatten(), plots
):
    print(f"Plotting {x_col} in the x-axis and {y_col} in the y-axis, coloured by {br_col} with the xlabel '{xlabel}', y-label '{ylabel}', and colorbar label '{cbarlabel}'")
    sc = ax.scatter(
        df_both[x_col],
        df_both[y_col],
        c=df_both[br_col],
        cmap='viridis',
        s=1.5,
        alpha=0.4,
        vmin=0,
        vmax=1,
        rasterized=True,
    )
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(cbarlabel, labelpad=8)
    ax.set_xlabel(xlabel, labelpad=8)
    ax.set_ylabel(ylabel, labelpad=8)
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.tick_params(direction='in', top=True, right=True)
    ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

plt.tight_layout()
save_fig('BR_parameter_planes.pdf')
plt.show()

## Downstream Decays and Phenomenology

In [ ]:
# df_both = df_both.copy()
# sp_cols = [c for c in df_both.columns if c.startswith('SP_BF_Z') or c.startswith('SP_BF_h_to')]
# df_both[sp_cols] = df_both[sp_cols].fillna(0)

# fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# # --- Z decay pie chart ---
# z_channels = {
#     r'$\ell^+\ell^-$': (
#         df_both['SP_BF_Z_to_ele_ele'] + 
#         df_both['SP_BF_Z_to_mu_mu'] + 
#         df_both['SP_BF_Z_to_tau_tau']
#     ).mean(),
#     r'$\nu\bar{\nu}$': (
#         df_both['SP_BF_Z_to_nu_ele_nu_ele'] + 
#         df_both['SP_BF_Z_to_nu_mu_nu_mu'] + 
#         df_both['SP_BF_Z_to_nu_tau_nu_tau']
#     ).mean(),
#     r'$q\bar{q}$': (
#         df_both['SP_BF_Z_to_d_d'] + 
#         df_both['SP_BF_Z_to_u_u'] + 
#         df_both['SP_BF_Z_to_s_s'] + 
#         df_both['SP_BF_Z_to_c_c'] + 
#         df_both['SP_BF_Z_to_b_b']
#     ).mean(),
#     r'Other': df_both['SP_BF_Z_other'].mean(),
# }

# # --- h decay pie chart ---
# h_channels = {
#     r'$b\bar{b}$': df_both['SP_BF_h_to_b_b'].mean(),
#     r'$WW^*$': df_both['SP_BF_h_to_W_W'].mean(),
#     r'$\tau^+\tau^-$': df_both['SP_BF_h_to_tau_tau'].mean(),
#     r'$gg$': df_both['SP_BF_h_to_g_g'].mean(),
#     r'$ZZ^*$': df_both['SP_BF_h_to_Z_Z'].mean(),
#     r'$\gamma\gamma$': df_both['SP_BF_h_to_gam_gam'].mean(),
#     r'$Z\gamma$': df_both['SP_BF_h_to_Z_gam'].mean(),
#     r'$\tilde{\chi}_1^0\tilde{\chi}_1^0$': df_both['SP_BF_h_to_chi_10_chi_10'].mean(),
#     r'Other': df_both['SP_BF_h_other'].mean(),
# }

# for ax, channels, title in zip(
#     axes,
#     [z_channels, h_channels],
#     [r'$Z$ boson decay branching fractions',
#      r'$h$ boson decay branching fractions']
# ):
#     labels = list(channels.keys())
#     values = list(channels.values())
#     wedges, texts, autotexts = ax.pie(
#         values,
#         labels=labels,
#         autopct=lambda p: f'{p:.1f}%%' if p > 1 else '',
#         startangle=90,
#         textprops={'fontsize': 11},
#     )
#     ax.set_title(title, fontsize=13)

# plt.tight_layout()
# # save_fig('decay_BRs_pie.pdf', bbox_inches='tight')
# plt.show()

In [ ]:
downstream_cols = [
    'SP_BF_Z_to_ele_ele', 'SP_BF_Z_to_mu_mu', 'SP_BF_Z_to_tau_tau',
    'SP_BF_Z_to_nu_ele_nu_ele', 'SP_BF_Z_to_nu_mu_nu_mu', 'SP_BF_Z_to_nu_tau_nu_tau',
    'SP_BF_Z_to_d_d', 'SP_BF_Z_to_u_u', 'SP_BF_Z_to_s_s', 'SP_BF_Z_to_c_c',
    'SP_BF_Z_to_b_b', 'SP_BF_Z_other',
    'SP_BF_h_to_s_s', 'SP_BF_h_to_c_c', 'SP_BF_h_to_b_b', 'SP_BF_h_to_mu_mu',
    'SP_BF_h_to_tau_tau', 'SP_BF_h_to_g_g', 'SP_BF_h_to_Z_gam', 'SP_BF_h_to_gam_gam',
    'SP_BF_h_to_Z_Z', 'SP_BF_h_to_W_W', 'SP_BF_h_to_chi_10_chi_10', 'SP_BF_h_other'
]

nonzero_counts = {col: (df_both[col] != 0).sum() for col in downstream_cols}
for col, count in nonzero_counts.items():
    print(f'{col}: {count}')

In [ ]:
z_channels = {
    r'$\ell^+\ell^-$': 0.1006,    # ~3.37% per lepton * 3
    r'$\nu\bar{\nu}$': 0.2000,    # ~6.67% per neutrino * 3
    r'$q\bar{q}$': 0.6991,        # hadronic
    r'Other': 0.0003,
}

In [ ]:
regimes = [
    (125.25, 250,  r'$m_h < \Delta m < 250$ GeV'),
    (250,    500,  r'$250 < \Delta m < 500$ GeV'),
    (500,    750,  r'$500 < \Delta m < 750$ GeV'),
    (750,    np.inf, r'$\Delta m > 750$ GeV'),
]

fig, axes = plt.subplots(1, 4, figsize=(24, 7))

for ax, (low, high, title) in zip(axes, regimes):
    mask = (df_both['Delta_m'] >= low) & (df_both['Delta_m'] < high)
    df_regime = df_both[mask]
    
    h_channels = {
        r'$b\bar{b}$':    df_regime['SP_BF_h_to_b_b'].mean(),
        r'$WW^*$':        df_regime['SP_BF_h_to_W_W'].mean(),
        r'$\tau^+\tau^-$': df_regime['SP_BF_h_to_tau_tau'].mean(),
        r'$gg$':          df_regime['SP_BF_h_to_g_g'].mean(),
        r'$ZZ^*$':        df_regime['SP_BF_h_to_Z_Z'].mean(),
        r'$\gamma\gamma$': df_regime['SP_BF_h_to_gam_gam'].mean(),
        r'$\tilde{\chi}_1^0\tilde{\chi}_1^0$': df_regime['SP_BF_h_to_chi_10_chi_10'].mean(),
        r'Other':         df_regime['SP_BF_h_other'].mean(),
    }

    # Drop negligible channels
    h_channels = {k: v for k, v in h_channels.items() if v > 0.005}

    labels = list(h_channels.keys())
    values = list(h_channels.values())

    ax.pie(
        values,
        labels=labels,
        autopct=lambda p: f'{p:.1f}%%' if p > 1 else '',
        startangle=90,
        textprops={'fontsize': 9},
        pctdistance=0.8,
    )
    ax.set_title(f'{title}\n(N={len(df_regime)})', fontsize=11)

plt.suptitle(r'$h$ boson decay branching fractions by $\Delta m$ regime', 
             fontsize=13, y=1.02)
plt.tight_layout()
# save_fig('h_decay_pie_regimes.pdf', bbox_inches='tight')
plt.show()

In [ ]:
regimes = [
    (125.25, 250,  r'$m_h < \Delta m < 250$ GeV'),
    (250,    500,  r'$250 < \Delta m < 500$ GeV'),
    (500,    750,  r'$500 < \Delta m < 750$ GeV'),
    (750,    np.inf, r'$\Delta m > 750$ GeV'),
]

fig, axes = plt.subplots(2, 2, figsize=(18, 18))
axes = axes.flatten()

for ax, (low, high, title) in zip(axes, regimes):
    mask = (df_both['Delta_m'] >= low) & (df_both['Delta_m'] < high)
    df_regime = df_both[mask]
    
    h_channels = {
        r'$b\bar{b}$':    df_regime['SP_BF_h_to_b_b'].mean(),
        r'$WW^*$':        df_regime['SP_BF_h_to_W_W'].mean(),
        r'$\tau^+\tau^-$': df_regime['SP_BF_h_to_tau_tau'].mean(),
        r'$gg$':          df_regime['SP_BF_h_to_g_g'].mean(),
        r'$ZZ^*$':        df_regime['SP_BF_h_to_Z_Z'].mean(),
        r'$\gamma\gamma$': df_regime['SP_BF_h_to_gam_gam'].mean(),
        r'$\tilde{\chi}_1^0\tilde{\chi}_1^0$': df_regime['SP_BF_h_to_chi_10_chi_10'].mean(),
        r'Other':         df_regime['SP_BF_h_other'].mean(),
    }

    # Drop negligible channels
    h_channels = {k: v for k, v in h_channels.items() if v > 0.005}

    labels = list(h_channels.keys())
    values = list(h_channels.values())

    ax.pie(
        values,
        labels=labels,
        autopct=lambda p: f'{p:.1f}%%' if p > 1 else '',
        startangle=90,
        textprops={'fontsize': 10},
        pctdistance=0.8,
        labeldistance=1.15,
    )
    ax.set_title(f'{title}\n(N={len(df_regime)})', fontsize=11)

plt.suptitle(r'$h$ boson decay branching fractions by $\Delta m$ regime', 
             fontsize=13, y=1.02)
plt.tight_layout()
# save_fig('h_decay_pie_regimes.pdf', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 16))
axes = axes.flatten()

for ax, (low, high, title) in zip(axes, regimes):
    mask = (df_both['Delta_m'] >= low) & (df_both['Delta_m'] < high)
    df_regime = df_both[mask]
    
    h_channels = {
        r'$b\bar{b}$':    df_regime['SP_BF_h_to_b_b'].mean(),
        r'$WW^*$':        df_regime['SP_BF_h_to_W_W'].mean(),
        r'$\tau^+\tau^-$': df_regime['SP_BF_h_to_tau_tau'].mean(),
        r'$gg$':          df_regime['SP_BF_h_to_g_g'].mean(),
        r'$ZZ^*$':        df_regime['SP_BF_h_to_Z_Z'].mean(),
        r'$\gamma\gamma$': df_regime['SP_BF_h_to_gam_gam'].mean(),
        r'$\tilde{\chi}_1^0\tilde{\chi}_1^0$': df_regime['SP_BF_h_to_chi_10_chi_10'].mean(),
        r'$c\bar{c}$': df_regime['SP_BF_h_to_c_c'].mean(),
        r'$s\bar{s}$': df_regime['SP_BF_h_to_s_s'].mean(),
        r'Other':         df_regime['SP_BF_h_other'].mean(),
    }
    
    keep_channels = {k for k, v in h_channels.items()}

    h_channels = {k: v for k, v in h_channels.items() if k in keep_channels}

    labels = list(h_channels.keys())
    values = list(h_channels.values())

    wedges, texts, autotexts = ax.pie(
        values,
        autopct=lambda p: f'{p:.1f}%%' if p > 1 else '',
        startangle=90,
        textprops={'fontsize': 11},
        pctdistance=0.75,
    )
    ax.legend(
        wedges, labels,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.25),
        fontsize=10,
        ncol=2,
    )
    ax.set_title(f'{title}\n(N={len(df_regime)})', fontsize=12)

# plt.suptitle(r'$h$ boson decay branching fractions by $\Delta m$ regime', fontsize=14, y=1.02)
plt.tight_layout()
# save_fig('h_decay_pie_regimes.pdf', bbox_inches='tight')
plt.show()

In [ ]:
h_channels = {
    r'$b\bar{b}$': df_both['SP_BF_h_to_b_b'].mean(),
    r'$WW^*$': df_both['SP_BF_h_to_W_W'].mean(),
    r'$\tau^+\tau^-$': df_both['SP_BF_h_to_tau_tau'].mean(),
    r'$gg$': df_both['SP_BF_h_to_g_g'].mean(),
    r'$ZZ^*$': df_both['SP_BF_h_to_Z_Z'].mean(),
    r'$\gamma\gamma$': df_both['SP_BF_h_to_gam_gam'].mean(),
    r'$\tilde{\chi}_1^0\tilde{\chi}_1^0$': df_both['SP_BF_h_to_chi_10_chi_10'].mean(),
    r'$c\bar{c}$': df_both['SP_BF_h_to_c_c'].mean(),
    r'$s\bar{s}$': df_both['SP_BF_h_to_s_s'].mean(),
    r'Other': df_both['SP_BF_h_other'].mean(),
}

# Drop negligible channels (< 0.5%)
h_channels = {k: v for k, v in h_channels.items() if v > 0.005}

labels = list(h_channels.keys())
values = list(h_channels.values())

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    values,
    labels=labels,
    autopct=lambda p: f'{p:.1f}%%' if p > 1 else '',
    startangle=90,
    textprops={'fontsize': 11},
    pctdistance=0.8,
)
# ax.set_title(r'$h$ boson decay branching fractions (model average)', fontsize=13)
plt.tight_layout()
# save_fig('h_decay_pie.pdf', bbox_inches='tight')
plt.show()

In [ ]:
h_channels = {
    r'$b\bar{b}$': df_both['SP_BF_h_to_b_b'].mean(),
    r'$WW^*$': df_both['SP_BF_h_to_W_W'].mean(),
    r'$\tau^+\tau^-$': df_both['SP_BF_h_to_tau_tau'].mean(),
    r'$gg$': df_both['SP_BF_h_to_g_g'].mean(),
    r'$ZZ^*$': df_both['SP_BF_h_to_Z_Z'].mean(),
    r'$\gamma\gamma$': df_both['SP_BF_h_to_gam_gam'].mean(),
    r'$\tilde{\chi}_1^0\tilde{\chi}_1^0$': df_both['SP_BF_h_to_chi_10_chi_10'].mean(),
    r'$c\bar{c}$': df_both['SP_BF_h_to_c_c'].mean(),
    r'$s\bar{s}$': df_both['SP_BF_h_to_s_s'].mean(),
    r'Other': df_both['SP_BF_h_other'].mean(),
}

# Drop negligible channels (< 0.5%)
h_channels = {k: v for k, v in h_channels.items() if v > 0.005}

labels = list(h_channels.keys())
values = list(h_channels.values())

# Use viridis colormap consistent with other plots
colors = plt.cm.viridis(np.linspace(0.05, 0.95, len(values)))

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    values,
    autopct=lambda p: f'{p:.1f}%%' if p > 1 else '',
    startangle=90,
    colors=colors,
    pctdistance=0.75,
    textprops={'fontsize': 12},
)

for autotext in autotexts:
    autotext.set_fontsize(12)

ax.legend(
    wedges, labels,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.15),
    fontsize=11,
    ncol=2,
    frameon=True,
)

plt.tight_layout()
# save_fig('h_decay_pie.pdf', bbox_inches='tight')
plt.show()

In [ ]:
h_channels = {
    r'$b\bar{b}$': df_both['SP_BF_h_to_b_b'].mean(),
    r'$WW^*$': df_both['SP_BF_h_to_W_W'].mean(),
    r'$\tau^+\tau^-$': df_both['SP_BF_h_to_tau_tau'].mean(),
    r'$gg$': df_both['SP_BF_h_to_g_g'].mean(),
    r'$ZZ^*$': df_both['SP_BF_h_to_Z_Z'].mean(),
    r'$\gamma\gamma$': df_both['SP_BF_h_to_gam_gam'].mean(),
    r'$\tilde{\chi}_1^0\tilde{\chi}_1^0$': df_both['SP_BF_h_to_chi_10_chi_10'].mean(),
    r'$c\bar{c}$': df_both['SP_BF_h_to_c_c'].mean(),
    r'$s\bar{s}$': df_both['SP_BF_h_to_s_s'].mean(),
    r'Other': df_both['SP_BF_h_other'].mean(),
}

h_channels = {k: v for k, v in h_channels.items() if v > 0.005}

labels = list(h_channels.keys())
values = list(h_channels.values())

# Use tab10 colormap for better contrast with black text
colors = plt.cm.tab10(np.linspace(0, 0.9, len(values)))

fig, ax = plt.subplots(figsize=(9,7))
wedges, texts, autotexts = ax.pie(
    values,
    labels=labels,
    autopct=lambda p: f'{p:.1f}%%' if p > 1 else '',
    startangle=90,
    colors=colors,
    pctdistance=0.75,
    labeldistance=1.1,
    textprops={'fontsize': 12},
)

for autotext in autotexts:
    autotext.set_fontsize(11)
    autotext.set_color('black')

for text in texts:
    text.set_fontsize(12)

plt.tight_layout()
plt.subplots_adjust(top=0.95, bottom=0.05)
save_fig('h_decay_pie.pdf')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

h_br_cols = {
    r'$b\bar{b}$': 'SP_BF_h_to_b_b',
    r'$WW^*$': 'SP_BF_h_to_W_W',
    r'$\tau^+\tau^-$': 'SP_BF_h_to_tau_tau',
    r'$gg$': 'SP_BF_h_to_g_g',
    r'$ZZ^*$': 'SP_BF_h_to_Z_Z',
    r'$\tilde{\chi}_1^0\tilde{\chi}_1^0$': 'SP_BF_h_to_chi_10_chi_10',
}

for ax, (label, col) in zip(axes, h_br_cols.items()):
    ax.hist(
        df_both[col],
        bins=50,
        color='steelblue',
        edgecolor='none',
        alpha=0.8,
        density=True,
    )
    ax.set_xlabel(f'BR({label})', labelpad=8)
    ax.set_ylabel('Probability density', labelpad=8)
    ax.tick_params(direction='in', top=True, right=True)
    ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

plt.tight_layout()
# save_fig('h_decay_BR_distributions.pdf', bbox_inches='tight')
plt.show()

In [ ]:
df_invisible = df_both[df_both['SP_BF_h_to_chi_10_chi_10'] > 0].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: vs Delta_m
sc0 = axes[0].scatter(
    df_invisible['Delta_m'],
    df_invisible['SP_BF_h_to_chi_10_chi_10'],
    c=df_invisible['SS_chi_20_Higgsino_frac'],
    cmap='viridis',
    s=3,
    alpha=0.5,
    rasterized=True,
)
cbar0 = fig.colorbar(sc0, ax=axes[0])
cbar0.set_label(r'Higgsino fraction of $\tilde{\chi}_2^0$', labelpad=8)
axes[0].set_xlabel(r'$\Delta m$ (GeV)', labelpad=8)
axes[0].set_ylabel(
    r'$\mathrm{BR}(h \to \tilde{\chi}_1^0\tilde{\chi}_1^0)$', labelpad=8
)
axes[0].tick_params(direction='in', top=True, right=True)
axes[0].grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

# Right: vs LSP mass
sc1 = axes[1].scatter(
    df_invisible['SP_m_chi_10'].abs(),
    df_invisible['SP_BF_h_to_chi_10_chi_10'],
    c=df_invisible['SS_chi_20_Higgsino_frac'],
    cmap='viridis',
    s=3,
    alpha=0.5,
    rasterized=True,
)
cbar1 = fig.colorbar(sc1, ax=axes[1])
cbar1.set_label(r'Higgsino fraction of $\tilde{\chi}_2^0$', labelpad=8)
axes[1].set_xlabel(r'$m_{\tilde{\chi}_1^0}$ (GeV)', labelpad=8)
axes[1].set_ylabel(
    r'$\mathrm{BR}(h \to \tilde{\chi}_1^0\tilde{\chi}_1^0)$', labelpad=8
)
axes[1].tick_params(direction='in', top=True, right=True)
axes[1].grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

plt.tight_layout()
save_fig('h_invisible_BR.pdf')
plt.show()

In [ ]:
print(f"Number of points with non-zero BR(h -> chi_10 chi_10): {len(df_invisible)}")
df_invisible.info()

In [ ]:
z_pdg = {
    r'ell+ell-  (per lepton)': 0.0337,
    r'nunu (per neutrino)':    0.0667,
    r'hadrons (total)':        0.6991,
}
print("SM Z decay branching fractions (PDG):")
for k, v in z_pdg.items():
    print(f"  {k}: {v*100:.1f}%")

## Exclusion Results

In [ ]:
valid = (df_analysis['SModelS_bestExpUL_TheoryPrediction'] > 0) & \
        (df_analysis['SModelS_bestExpUL_UpperLimit'] > 0)

df_analysis['excluded'] = valid & \
    (df_analysis['SModelS_bestExpUL_TheoryPrediction'] > 
     df_analysis['SModelS_bestExpUL_UpperLimit'])

print(f"Excluded: {df_analysis['excluded'].sum()}")
print(f"Surviving: {(~df_analysis['excluded']).sum()}")
print(f"Exclusion fraction: {df_analysis['excluded'].mean():.3f}")

In [ ]:
print(df_analysis['SModelS_bestExpUL_TheoryPrediction'].value_counts().head(10))

In [ ]:
df_analysis['excluded_exp'] = valid & \
    (df_analysis['SModelS_bestExpUL_TheoryPrediction'] > 
     df_analysis['SModelS_bestExpUL_ExpUpperLimit'])

print(f"Excluded (exp): {df_analysis['excluded_exp'].sum()}")

In [ ]:
def exclusion_heatmap(ax, x, y, excluded, xlabel, ylabel, 
                      xbins=20, ybins=20, cmap='viridis'):
    """Plot exclusion fraction as a 2D heatmap."""
    
    stat, xedges, yedges, _ = binned_statistic_2d(
        x, y, excluded.astype(float),
        statistic='mean',
        bins=[xbins, ybins]
    )
    
    counts, _, _, _ = binned_statistic_2d(
        x, y, excluded.astype(float),
        statistic='count',
        bins=[xedges, yedges]
    )
    
    # Mask bins with no models
    stat = np.ma.masked_where(counts == 0, stat)
    
    im = ax.pcolormesh(
        xedges, yedges, stat.T,
        cmap=cmap,
        vmin=0, vmax=1,
    )
    ax.set_xlabel(xlabel, labelpad=8)
    ax.set_ylabel(ylabel, labelpad=8)
    ax.tick_params(direction='in', top=True, right=True)
    ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.4)
    return im

# --- Use df_both with exclusion flag ---
df_both = df_analysis[df_analysis['Delta_m'] >= 125.25].copy()
# df_plot = df_both.copy()
df_plot = df_analysis.copy()

# --- Figure 1: Mass plane ---
fig, ax = plt.subplots(figsize=(8, 6))
im = exclusion_heatmap(
    ax,
    df_plot['SP_m_chi_10'].abs(),
    df_plot['SP_m_chi_20'].abs(),
    df_plot['excluded_exp'],
    xlabel=r'$m_{\tilde{\chi}_1^0}$ (GeV)',
    ylabel=r'$m_{\tilde{\chi}_2^0}$ (GeV)',
)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Fraction of models excluded', labelpad=8)
plt.tight_layout()
# save_fig('exclusion_mass_plane.pdf', bbox_inches='tight')
plt.show()

# --- Figure 2: M1-mu and M2-mu planes side by side ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = exclusion_heatmap(
    axes[0],
    df_plot['SP_M_1'].abs(),
    df_plot['SP_mu'].abs(),
    df_plot['excluded_exp'],
    xlabel=r'$|M_1|$ (GeV)',
    ylabel=r'$|\mu|$ (GeV)',
)
cbar0 = fig.colorbar(im0, ax=axes[0])
cbar0.set_label('Fraction of models excluded', labelpad=8)

im1 = exclusion_heatmap(
    axes[1],
    df_plot['SP_M_2'].abs(),
    df_plot['SP_mu'].abs(),
    df_plot['excluded_exp'],
    xlabel=r'$|M_2|$ (GeV)',
    ylabel=r'$|\mu|$ (GeV)',
)
cbar1 = fig.colorbar(im1, ax=axes[1])
cbar1.set_label('Fraction of models excluded', labelpad=8)

plt.tight_layout()
# save_fig('exclusion_parameter_planes.pdf', bbox_inches='tight')
plt.show()

# --- Figure 3: Delta_m vs m_chi10 plane ---
fig, ax = plt.subplots(figsize=(8, 6))
im = exclusion_heatmap(
    ax,
    df_plot['SP_m_chi_10'].abs(),
    df_plot['Delta_m'],
    df_plot['excluded_exp'],
    xlabel=r'$m_{\tilde{\chi}_1^0}$ (GeV)',
    ylabel=r'$\Delta m$ (GeV)',
)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Fraction of models excluded', labelpad=8)
plt.tight_layout()
# save_fig('exclusion_deltam_plane.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# --- Option 1: Log scale colour ---
fig, ax = plt.subplots(figsize=(8, 6))
stat, xedges, yedges, _ = binned_statistic_2d(
    df_plot['SP_m_chi_10'].abs(),
    df_plot['SP_m_chi_20'].abs(),
    df_plot['excluded_exp'].astype(float),
    statistic='mean',
    bins=20
)
counts, _, _, _ = binned_statistic_2d(
    df_plot['SP_m_chi_10'].abs(),
    df_plot['SP_m_chi_20'].abs(),
    df_plot['excluded_exp'].astype(float),
    statistic='count',
    bins=[xedges, yedges]
)
stat = np.ma.masked_where(counts == 0, stat)
stat_log = np.ma.masked_where(stat <= 0, stat)

im = ax.pcolormesh(
    xedges, yedges, stat_log.T,
    cmap='viridis',
    norm=mcolors.LogNorm(vmin=0.01, vmax=1),
)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Fraction of models excluded (log scale)', labelpad=8)
ax.set_xlabel(r'$m_{\tilde{\chi}_1^0}$ (GeV)', labelpad=8)
ax.set_ylabel(r'$m_{\tilde{\chi}_2^0}$ (GeV)', labelpad=8)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.4)
ax.set_title('Option 1: Log scale')
plt.tight_layout()
plt.show()

# --- Option 2: plasma colormap ---
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.pcolormesh(
    xedges, yedges, stat.T,
    cmap='plasma',
    vmin=0, vmax=1,
)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Fraction of models excluded', labelpad=8)
ax.set_xlabel(r'$m_{\tilde{\chi}_1^0}$ (GeV)', labelpad=8)
ax.set_ylabel(r'$m_{\tilde{\chi}_2^0}$ (GeV)', labelpad=8)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.4)
ax.set_title('Option 2: Plasma colormap')
plt.tight_layout()
plt.show()

# --- Option 3: r-value (theory/upper limit) ---
valid = (df_plot['SModelS_bestExpUL_TheoryPrediction'] > 0) & \
        (df_plot['SModelS_bestExpUL_ExpUpperLimit'] > 0)
df_plot['r_value'] = np.where(
    valid,
    df_plot['SModelS_bestExpUL_TheoryPrediction'] / 
    df_plot['SModelS_bestExpUL_ExpUpperLimit'],
    np.nan
)

r_stat, _, _, _ = binned_statistic_2d(
    df_plot['SP_m_chi_10'].abs(),
    df_plot['SP_m_chi_20'].abs(),
    df_plot['r_value'],
    statistic='mean',
    bins=[xedges, yedges]
)
r_stat = np.ma.masked_where(counts == 0, r_stat)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.pcolormesh(
    xedges, yedges, r_stat.T,
    cmap='viridis',
    norm=mcolors.LogNorm(vmin=0.01, vmax=2),
)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label(r'Mean $r = \sigma_\mathrm{theory}/\sigma_\mathrm{UL}$', labelpad=8)
ax.set_xlabel(r'$m_{\tilde{\chi}_1^0}$ (GeV)', labelpad=8)
ax.set_ylabel(r'$m_{\tilde{\chi}_2^0}$ (GeV)', labelpad=8)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.4)
ax.set_title('Option 3: r-value')
plt.tight_layout()
plt.show()

In [ ]:
# --- Use df_analysis ---
df_excl = df_analysis.copy()

def exclusion_heatmap_log(ax, x, y, excluded, xlabel, ylabel,
                          xbins=20, ybins=20):
    stat, xedges, yedges, _ = binned_statistic_2d(
        x, y, excluded.astype(float),
        statistic='mean',
        bins=[xbins, ybins]
    )
    counts, _, _, _ = binned_statistic_2d(
        x, y, excluded.astype(float),
        statistic='count',
        bins=[xedges, yedges]
    )
    stat = np.ma.masked_where((counts == 0) | (stat <= 0), stat)
    
    im = ax.pcolormesh(
        xedges, yedges, stat.T,
        cmap='viridis',
        norm=mcolors.LogNorm(vmin=0.01, vmax=1),
    )
    ax.set_xlabel(xlabel, labelpad=8)
    ax.set_ylabel(ylabel, labelpad=8)
    ax.tick_params(direction='in', top=True, right=True)
    ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.4)
    return im

# --- Figure 1: Mass plane ---
fig, ax = plt.subplots(figsize=(8, 6))
im = exclusion_heatmap_log(
    ax,
    df_excl['SP_m_chi_10'].abs(),
    df_excl['SP_m_chi_20'].abs(),
    df_excl['excluded_exp'],
    xlabel=r'$m_{\tilde{\chi}_1^0}$ (GeV)',
    ylabel=r'$m_{\tilde{\chi}_2^0}$ (GeV)',
)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Fraction of models excluded', labelpad=8)
plt.tight_layout()
# save_fig('exclusion_mass_plane.pdf', bbox_inches='tight')
plt.show()

# --- Figure 2: M1-mu and M2-mu planes side by side ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = exclusion_heatmap_log(
    axes[0],
    df_excl['SP_M_1'].abs(),
    df_excl['SP_mu'].abs(),
    df_excl['excluded_exp'],
    xlabel=r'$|M_1|$ (GeV)',
    ylabel=r'$|\mu|$ (GeV)',
)
cbar0 = fig.colorbar(im0, ax=axes[0])
cbar0.set_label('Fraction of models excluded', labelpad=8)

im1 = exclusion_heatmap_log(
    axes[1],
    df_excl['SP_M_2'].abs(),
    df_excl['SP_mu'].abs(),
    df_excl['excluded_exp'],
    xlabel=r'$|M_2|$ (GeV)',
    ylabel=r'$|\mu|$ (GeV)',
)
cbar1 = fig.colorbar(im1, ax=axes[1])
cbar1.set_label('Fraction of models excluded', labelpad=8)

plt.tight_layout()
# save_fig('exclusion_parameter_planes.pdf', bbox_inches='tight')
plt.show()

# --- Figure 3: Delta_m vs m_chi10 plane ---
fig, ax = plt.subplots(figsize=(8, 6))
im = exclusion_heatmap_log(
    ax,
    df_excl['SP_m_chi_10'].abs(),
    df_excl['Delta_m'],
    df_excl['excluded_exp'],
    xlabel=r'$m_{\tilde{\chi}_1^0}$ (GeV)',
    ylabel=r'$\Delta m$ (GeV)',
)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Fraction of models excluded', labelpad=8)
plt.tight_layout()
# save_fig('exclusion_deltam_plane.pdf', bbox_inches='tight')
plt.show()

In [ ]:
def exclusion_heatmap_log(ax, x, y, excluded, xlabel, ylabel,
                          xbins=12, ybins=12):
    stat, xedges, yedges, _ = binned_statistic_2d(
        x, y, excluded.astype(float),
        statistic='mean',
        bins=[xbins, ybins]
    )
    counts, _, _, _ = binned_statistic_2d(
        x, y, excluded.astype(float),
        statistic='count',
        bins=[xedges, yedges]
    )

    # Grey background for bins with models but no exclusions
    zero_mask = np.where((counts > 0) & (stat == 0), 1, np.nan)
    ax.pcolormesh(
        xedges, yedges, zero_mask.T,
        cmap='Greys',
        vmin=0, vmax=2,
    )

    # White for bins with no models at all (default background)
    # Colour scale for bins with at least one excluded model
    stat_plot = np.where((counts > 0) & (stat > 0), stat, np.nan)

    im = ax.pcolormesh(
        xedges, yedges, stat_plot.T,
        cmap='viridis',
        norm=mcolors.LogNorm(vmin=0.01, vmax=1),
    )

    ax.set_xlabel(xlabel, labelpad=8)
    ax.set_ylabel(ylabel, labelpad=8)
    ax.tick_params(direction='in', top=True, right=True)
    ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.4)
    return im

# --- Figure 1: Mass plane ---
fig, ax = plt.subplots(figsize=(8, 6))
im = exclusion_heatmap_log(
    ax,
    df_excl['SP_m_chi_10'].abs(),
    df_excl['SP_m_chi_20'].abs(),
    df_excl['excluded_exp'],
    xlabel=r'$m_{\tilde{\chi}_1^0}$ (GeV)',
    ylabel=r'$m_{\tilde{\chi}_2^0}$ (GeV)',
)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Fraction of models excluded', labelpad=8)
plt.tight_layout()
save_fig('exclusion_mass_plane.pdf')
plt.show()

# --- Figure 2: M1-mu and M2-mu planes side by side ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = exclusion_heatmap_log(
    axes[0],
    df_excl['SP_M_1'].abs(),
    df_excl['SP_mu'].abs(),
    df_excl['excluded_exp'],
    xlabel=r'$|M_1|$ (GeV)',
    ylabel=r'$|\mu|$ (GeV)',
)
cbar0 = fig.colorbar(im0, ax=axes[0])
cbar0.set_label('Fraction of models excluded', labelpad=8)

im1 = exclusion_heatmap_log(
    axes[1],
    df_excl['SP_M_2'].abs(),
    df_excl['SP_mu'].abs(),
    df_excl['excluded_exp'],
    xlabel=r'$|M_2|$ (GeV)',
    ylabel=r'$|\mu|$ (GeV)',
)
cbar1 = fig.colorbar(im1, ax=axes[1])
cbar1.set_label('Fraction of models excluded', labelpad=8)

plt.tight_layout()
save_fig('exclusion_parameter_planes.pdf')
plt.show()

# --- Figure 3: Delta_m vs m_chi10 plane ---
fig, ax = plt.subplots(figsize=(8, 6))
im = exclusion_heatmap_log(
    ax,
    df_excl['SP_m_chi_10'].abs(),
    df_excl['Delta_m'],
    df_excl['excluded_exp'],
    xlabel=r'$m_{\tilde{\chi}_1^0}$ (GeV)',
    ylabel=r'$\Delta m$ (GeV)',
)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Fraction of models excluded', labelpad=8)
plt.tight_layout()
save_fig('exclusion_deltam_plane.pdf')
plt.show()

# Fine-Tuning Analysis

In [ ]:
# Distribution of Delta_EW

plt.figure(figsize=(10, 6))
sns.histplot(
    df_analysis['Delta_EW'],
    bins=50,
    kde=True,
    color='steelblue',
    edgecolor='white',
    linewidth=0.4,
    alpha=0.75
)

p16 = np.percentile(df_analysis['Delta_EW'], 16)
p50 = np.median(df_analysis['Delta_EW'])
p84 = np.percentile(df_analysis['Delta_EW'], 84)
n = len(df_analysis)

plt.text(
    0.95, 0.92,
    f'$N = {n}$\n'
    f'16th: {p16:.2f}\n'
    f'Median: {p50:.2f}\n'
    f'84th: {p84:.2f}',
    transform=plt.gca().transAxes,
    ha='right', va='top',
    fontsize=11,
    bbox=dict(boxstyle='round,pad=0.4', fc='white',
              ec='gray', alpha=0.9, lw=0.8)
)

plt.xlabel(r'$\Delta_{\mathrm{EW}}$', labelpad=8)
plt.ylabel('Number of models', labelpad=8)
# plt.title(r'Distribution of $\Delta_{\mathrm{EW}}$ for surviving models', pad=10)
plt.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(
    df_analysis['Delta_EW'],
    bins=50,
    color='steelblue',
    edgecolor='white',
    linewidth=0.4,
    alpha=0.75
)


kde_vals = df_analysis['Delta_EW'].dropna()
kde = gaussian_kde(kde_vals, bw_method=0.3)
x_range = np.linspace(kde_vals.min(), kde_vals.max(), 500)
ax.plot(x_range, kde(x_range) * len(kde_vals) * (kde_vals.max() - kde_vals.min()) / 50,
        color='steelblue', linewidth=1.5)

p16 = np.percentile(df_analysis['Delta_EW'].dropna(), 16)
p50 = np.median(df_analysis['Delta_EW'].dropna())
p84 = np.percentile(df_analysis['Delta_EW'].dropna(), 84)
n = df_analysis['Delta_EW'].notna().sum()

ax.text(
    0.95, 0.92,
    f'$N = {n}$\n'
    f'16th: {p16:.2f}\n'
    f'Median: {p50:.2f}\n'
    f'84th: {p84:.2f}',
    transform=ax.transAxes,
    ha='right', va='top',
    fontsize=11,
    bbox=dict(boxstyle='round,pad=0.4', fc='white',
              ec='gray', alpha=0.9, lw=0.8)
)

ax.axvline(30, color='red', linestyle='--', linewidth=1.2, 
           label=r'$\Delta_{\mathrm{EW}} = 30$')
ax.legend(fontsize=11)

ax.set_xlim(0, None)
ax.set_xlabel(r'$\Delta_{\mathrm{EW}}$', labelpad=8)
ax.set_ylabel('Number of models', labelpad=8)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

plt.tight_layout()
save_fig('delta_ew_distribution.pdf')
plt.show()

In [ ]:
peak_bin = np.histogram(df_analysis['Delta_EW'].dropna(), bins=50)
peak_idx = np.argmax(peak_bin[0])
peak_value = (peak_bin[1][peak_idx] + peak_bin[1][peak_idx + 1]) / 2
print(f"Peak of distribution: Delta_EW ~ {peak_value:.1f}")

In [ ]:
# Compute KDE
data = df_analysis['Delta_EW'].dropna()
kde = gaussian_kde(data)

# Find the mode (peak of KDE)
x_range = np.linspace(data.min(), data.max(), 10000)
kde_values = kde(x_range)
mode = x_range[np.argmax(kde_values)]

print(f"Mode (KDE peak): {mode:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.histplot(
    df_analysis['Delta_EW'],
    bins=50,
    kde=True,
    color='steelblue',
    edgecolor='white',
    linewidth=0.4,
    alpha=0.8,
    line_kws={'linewidth': 1.5, 'color': 'navy'},
    ax=ax,
    stat='density'
)

p16 = np.percentile(df_analysis['Delta_EW'], 16)
p50 = np.median(df_analysis['Delta_EW'])
p84 = np.percentile(df_analysis['Delta_EW'], 84)
n = len(df_analysis)

ax.text(
    0.95, 0.92,
    f'$N = {n}$\n'
    f'16th: {p16:.2f}\n'
    f'Median: {p50:.2f}\n'
    f'84th: {p84:.2f}',
    transform=ax.transAxes,
    ha='right', va='top',
    fontsize=11,
    bbox=dict(boxstyle='round,pad=0.4', fc='white',
              ec='gray', alpha=0.9, lw=0.8)
)

# ax.set_xscale('log')
ax.set_xlabel(r'$\Delta_{\mathrm{EW}}$', labelpad=8)
ax.set_ylabel(r'Probability density', labelpad=8)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)

plt.tight_layout()
# save_fig('Delta_EW_distribution.pdf', bbox_inches='tight')
plt.show()

In [ ]:
for threshold in [10, 30, 100, 1000]:
    n = (df_analysis['Delta_EW'] < threshold).sum()
    pct = 100 * n / len(df_analysis)
    print(f"Delta_EW < {threshold}: {n} ({pct:.1f}%)")

In [ ]:
# Compute KDE
data = df_analysis['Delta_EW'].dropna()
kde = gaussian_kde(data)

# Find the mode (peak of KDE)
x_range = np.linspace(data.min(), data.max(), 10000)
kde_values = kde(x_range)
mode = x_range[np.argmax(kde_values)]

print(f"Mode (KDE peak): {mode:.2f}")

In [ ]:
# Delta_EW vs mu_abs

plt.figure(figsize=(10, 6))
plt.scatter(
    df_analysis['mu_abs'],
    df_analysis['Delta_EW'],
    alpha=0.6,
    edgecolor='k',
    label='Surviving Models',
    marker='x'
)
plt.xlabel(r'$|\mu|$ (GeV)', labelpad=8)
plt.ylabel(r'$\Delta_{\mathrm{EW}}$', labelpad=8)
plt.title(r'$\Delta_{\mathrm{EW}}$ vs $|\mu|$ for surviving models', pad=10)
plt.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
# plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(
    df_analysis['mu_abs'],
    df_analysis['Delta_EW'],
    s=1.5,
    alpha=0.4,
    color='steelblue',
    rasterized=True,
)

ax.axhline(30, color='red', linestyle='--', linewidth=1.2,
           label=r'$\Delta_{\mathrm{EW}} = 30$')
ax.set_xlabel(r'$|\mu|$ (GeV)', labelpad=8)
ax.set_ylabel(r'$\Delta_{\mathrm{EW}}$', labelpad=8)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
ax.set_xlim(0, None)
ax.set_ylim(0, None)
ax.legend(fontsize=11)

plt.tight_layout()
save_fig('delta_ew_vs_mu.pdf')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(
    df_analysis['mu_abs']**2,
    df_analysis['Delta_EW'],
    alpha=0.6,
    edgecolor='k',
    label='Surviving Models',
    marker='x'
)
plt.xlabel(r'$|\mu|^2$ (GeV$^2$)', labelpad=8)
plt.ylabel(r'$\Delta_{\mathrm{EW}}$', labelpad=8)
plt.title(r'$\Delta_{\mathrm{EW}}$ vs $|\mu|^2$ for surviving models', pad=10)
plt.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
# plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ΔEW​ vs ∥M1∥

plt.figure(figsize=(10, 6))
plt.scatter(
    df_analysis['M1_abs'],
    df_analysis['Delta_EW'],
    alpha=0.6,
    edgecolor='k',
    label='Surviving Models',
    marker='x'
)
plt.xlabel(r'$|M_1|$ (GeV)', labelpad=8)
plt.ylabel(r'$\Delta_{\mathrm{EW}}$', labelpad=8)
plt.title(r'$\Delta_{\mathrm{EW}}$ vs $|M_1|$ for surviving models', pad=10)
plt.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
# plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ΔEW​ vs ∥M2∥

plt.figure(figsize=(10, 6))
plt.scatter(
    df_analysis['M2_abs'],
    df_analysis['Delta_EW'],
    alpha=0.6,
    edgecolor='k',
    label='Surviving Models',
    marker='x'
)
plt.xlabel(r'$|M_2|$ (GeV)', labelpad=8)
plt.ylabel(r'$\Delta_{\mathrm{EW}}$', labelpad=8)
plt.title(r'$\Delta_{\mathrm{EW}}$ vs $|M_2|$ for surviving models', pad=10)
plt.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
# plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ΔEW​ vs BR(Z) and BR(h) (side by side)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)

for ax, br_col, ylabel in zip(
    axes,
    ['SP_BF_chi_20_to_chi_10_Z', 'SP_BF_chi_20_to_chi_10_h'],
    [r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$',
     r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$'],
):
    sc = ax.scatter(
        df_analysis[br_col],
        df_analysis['Delta_EW'],
        alpha=0.6,
        edgecolor='k',
        label='Surviving Models',
        marker='x'
    )
    ax.set_xlabel(ylabel, labelpad=8)
    ax.set_ylabel(r'$\Delta_{\mathrm{EW}}$', labelpad=8)
    ax.set_title(f'$\Delta_{{\mathrm{{EW}}}}$ vs {ylabel}', pad=10)
    ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
    # ax.legend()
    
plt.tight_layout()
plt.show()

In [ ]:
# ΔEW​ in ∥M1∥--∥mu∥ plane

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(
    df_analysis['M1_abs'],
    df_analysis['mu_abs'],
    c=df_analysis['Delta_EW'],
    cmap='viridis',
    s=1.5,
    alpha=0.6,
    vmin=df_analysis['Delta_EW'].min(),
    vmax=df_analysis['Delta_EW'].max(),
    rasterized=True
)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(r'$\Delta_{\mathrm{EW}}$', labelpad=8)
ax.set_xlabel(r'$|M_1|$ (GeV)', labelpad=8)
ax.set_ylabel(r'$|\mu|$ (GeV)', labelpad=8)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# ΔEW​ vs higgsino fraction of χ~20\tilde{\chi}_2^0

plt.figure(figsize=(10, 6))

plt.scatter(
    df_analysis['SP_chi_20_Higgsino_frac'],
    df_analysis['Delta_EW'],
    alpha=0.6,
    edgecolor='k',
    label='Surviving Models',
    marker='x'
)
plt.xlabel(r'Higgsino fraction of $\tilde{\chi}_2^0$', labelpad=8)
plt.ylabel(r'$\Delta_{\mathrm{EW}}$', labelpad=8)
plt.title(r'$\Delta_{\mathrm{EW}}$ vs Higgsino fraction of $\tilde{\chi}_2^0$', pad=10)
plt.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
# plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

BR_Z = 'SP_BF_chi_20_to_chi_10_Z'
BR_h = 'SP_BF_chi_20_to_chi_10_h'

for ax, br_col, label in zip(
    axes,
    [BR_Z, BR_h],
    [r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$',
     r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$']
):
    sc = ax.scatter(
        df_analysis['mu_abs'],
        df_analysis['Delta_EW'],
        c=df_analysis[br_col],
        cmap='viridis',
        s=1.5,
        alpha=0.4,
        vmin=0,
        vmax=1,
        rasterized=True,
    )
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(label, labelpad=8)
    ax.set_xlabel(r'$|\mu|$ (GeV)', labelpad=8)
    ax.set_ylabel(r'$\Delta_{\mathrm{EW}}$', labelpad=8)
    ax.tick_params(direction='in', top=True, right=True)
    ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
    ax.set_xlim(0, None)
    ax.set_ylim(0, None)

plt.tight_layout()
# save_fig('delta_ew_vs_mu_BR.pdf', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

BR_Z = 'SP_BF_chi_20_to_chi_10_Z'
BR_h = 'SP_BF_chi_20_to_chi_10_h'

for ax, br_col, xlabel in zip(
    axes,
    [BR_Z, BR_h],
    [r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 Z)$',
     r'$\mathrm{BR}(\tilde{\chi}_2^0 \to \tilde{\chi}_1^0 h)$']
):
    sc = ax.scatter(
    df_analysis[br_col],
    df_analysis['Delta_EW'],
    c=df_analysis['mu_abs'],
    cmap='viridis',
    s=1.5,
    alpha=0.4,
    # norm=LogNorm(),
    rasterized=True,
    )
    
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(r'$|\mu|$ (GeV)', labelpad=8)
    ax.set_xlabel(xlabel, labelpad=8)
    ax.set_ylabel(r'$\Delta_{\mathrm{EW}}$', labelpad=8)
    ax.tick_params(direction='in', top=True, right=True)
    ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, None)

plt.tight_layout()
# save_fig('delta_ew_vs_BR.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Bin Delta_EW and compute exclusion fraction in each bin
bins = np.linspace(0, 1000, 21)  # 20 bins from 0 to 1000
bin_centers = (bins[:-1] + bins[1:]) / 2

excl_fracs = []
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (df_analysis['Delta_EW'] >= lo) & (df_analysis['Delta_EW'] < hi)
    subset = df_analysis[mask]
    if len(subset) == 0:
        excl_fracs.append(np.nan)
    else:
        excl_frac = subset['excluded'].mean()
        excl_fracs.append(excl_frac)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(bin_centers, excl_fracs, 
        color='steelblue', linewidth=1.5, marker='o', markersize=4)
ax.axvline(30, color='red', linestyle='--', linewidth=1.2,
           label=r'$\Delta_{\mathrm{EW}} = 30$')

ax.set_xlabel(r'$\Delta_{\mathrm{EW}}$', labelpad=8)
ax.set_ylabel('Fraction of models excluded', labelpad=8)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
ax.set_xlim(0, None)
ax.set_ylim(0, None)
ax.legend(fontsize=11)

plt.tight_layout()
# save_fig('delta_ew_vs_excl.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Compute excluded column
df_analysis['excluded'] = (
    df_analysis['SModelS_bestExpUL_TheoryPrediction'] > 
    df_analysis['SModelS_bestExpUL_UpperLimit']
).astype(int)

# Percentile-based bins
bins = np.percentile(df_analysis['Delta_EW'].dropna(), np.linspace(0, 100, 21))
bins = np.unique(bins)
bin_centers = (bins[:-1] + bins[1:]) / 2

excl_fracs = []
n_models = []
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (df_analysis['Delta_EW'] >= lo) & (df_analysis['Delta_EW'] < hi)
    subset = df_analysis[mask]
    if len(subset) == 0:
        excl_fracs.append(np.nan)
        n_models.append(0)
    else:
        excl_fracs.append(subset['excluded'].mean())
        n_models.append(len(subset))

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(bin_centers, excl_fracs,
        color='steelblue', linewidth=1.5, marker='o', markersize=4)
ax.axvline(30, color='red', linestyle='--', linewidth=1.2,
           label=r'$\Delta_{\mathrm{EW}} = 30$')

ax.set_xlabel(r'$\Delta_{\mathrm{EW}}$', labelpad=8)
ax.set_ylabel('Fraction of models excluded', labelpad=8)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
ax.set_xlim(0, None)
ax.set_ylim(0, None)
ax.legend(fontsize=11)

plt.tight_layout()
save_fig('delta_ew_vs_excl.pdf')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sc = ax.scatter(
    df_analysis['SP_m_chi_10'].abs(),
    df_analysis['Delta_EW'],
    s=1.5,
    alpha=0.4,
    color='steelblue',
    rasterized=True,
)

ax.axhline(30, color='red', linestyle='--', linewidth=1.2,
           label=r'$\Delta_{\mathrm{EW}} = 30$')
ax.legend(fontsize=11)
ax.set_xlabel(r'$m_{\tilde{\chi}_1^0}$ (GeV)', labelpad=8)
ax.set_ylabel(r'$\Delta_{\mathrm{EW}}$', labelpad=8)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
ax.set_xlim(0, None)
ax.set_ylim(0, None)

plt.tight_layout()
save_fig('delta_ew_vs_mchi10.pdf')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sc = ax.scatter(
    df_analysis['SP_m_t_1'].abs(),
    df_analysis['Delta_EW'],
    s=1.5,
    alpha=0.4,
    color='steelblue',
    rasterized=True,
)

ax.axhline(30, color='red', linestyle='--', linewidth=1.2,
           label=r'$\Delta_{\mathrm{EW}} = 30$')
ax.legend(fontsize=11)
ax.set_xlabel(r'$m_{\tilde{t}_1}$ (GeV)', labelpad=8)
ax.set_ylabel(r'$\Delta_{\mathrm{EW}}$', labelpad=8)
ax.tick_params(direction='in', top=True, right=True)
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.7)
# ax.set_xlim(0, None)
ax.set_ylim(0, None)

plt.tight_layout()
save_fig('delta_ew_vs_mst1.pdf')
plt.show()

In [ ]:
# Determine which C_i dominates for each model
C_mu_vals     = df_analysis['SP_mu']**2
C_mHu_vals    = (df_analysis['mHu2'] * df_analysis['SP_tanb_min']**2 / 
                 (df_analysis['SP_tanb_min']**2 - 1)).abs()
C_mHd_vals    = (df_analysis['mHd2'] / 
                 (df_analysis['SP_tanb_min']**2 - 1)).abs()
C_Suu_vals    = df_analysis['C_Sigma_uu'] if 'C_Sigma_uu' in df_analysis.columns \
                else pd.Series(np.zeros(len(df_analysis)))
C_Sdd_vals    = df_analysis['C_Sigma_dd'] if 'C_Sigma_dd' in df_analysis.columns \
                else pd.Series(np.zeros(len(df_analysis)))

C_df = pd.DataFrame({
    r'$C_\mu$': C_mu_vals.values,
    r'$C_{H_u}$': C_mHu_vals.values,
    r'$C_{H_d}$': C_mHd_vals.values,
    r'$C_{\Sigma_u^u}$': C_Suu_vals.values,
    r'$C_{\Sigma_d^d}$': C_Sdd_vals.values,
})

dominant = C_df.idxmax(axis=1)
counts = dominant.value_counts()

fig, ax = plt.subplots(figsize=(8, 8))
# colors = plt.cm.viridis(np.linspace(0.05, 0.95, len(counts)))
# colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']
colors = plt.cm.tab10(np.linspace(0, 0.9, len(counts)))

wedges, texts, autotexts = ax.pie(
    counts.values,
    labels=counts.index,
    autopct=lambda p: f'{p:.1f}%%' if p > 1 else '',
    startangle=90,
    colors=colors,
    pctdistance=0.75,
    textprops={'fontsize': 12},
)
for autotext in autotexts:
    autotext.set_fontsize(11)

plt.tight_layout()
# save_fig('delta_ew_dominant_term.pdf', bbox_inches='tight')
plt.show()

print(counts)

In [ ]:
# Determine which C_i dominates for each model
C_mu_vals     = df_analysis['SP_mu']**2
C_mHu_vals    = (df_analysis['mHu2'] * df_analysis['SP_tanb_min']**2 / 
                 (df_analysis['SP_tanb_min']**2 - 1)).abs()
C_mHd_vals    = (df_analysis['mHd2'] / 
                 (df_analysis['SP_tanb_min']**2 - 1)).abs()
C_Suu_vals    = df_analysis['C_Sigma_uu'] if 'C_Sigma_uu' in df_analysis.columns \
                else pd.Series(np.zeros(len(df_analysis)))
C_Sdd_vals    = df_analysis['C_Sigma_dd'] if 'C_Sigma_dd' in df_analysis.columns \
                else pd.Series(np.zeros(len(df_analysis)))

C_df = pd.DataFrame({
    r'$C_\mu$': C_mu_vals.values,
    r'$C_{H_u}$': C_mHu_vals.values,
    r'$C_{H_d}$': C_mHd_vals.values,
    r'$C_{\Sigma_u^u}$': C_Suu_vals.values,
    r'$C_{\Sigma_d^d}$': C_Sdd_vals.values,
})

dominant = C_df.idxmax(axis=1)
counts = dominant.value_counts()

fig, ax = plt.subplots(figsize=(8, 8))
# colors = plt.cm.viridis(np.linspace(0.05, 0.95, len(counts)))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']
# colors = plt.cm.tab10(np.linspace(0, 0.9, len(counts)))

wedges, texts, autotexts = ax.pie(
    counts.values,
    labels=counts.index,
    autopct=lambda p: f'{p:.1f}%%' if p > 1 else '',
    startangle=90,
    colors=colors,
    pctdistance=0.75,
    textprops={'fontsize': 12},
)
for autotext in autotexts:
    autotext.set_fontsize(11)

plt.tight_layout()
save_fig('delta_ew_dominant_term.pdf')
plt.show()

print(counts)

In [ ]:
print("Dominant C_i term breakdown:")
print(counts.to_string())
print(f"\nTotal models: {counts.sum()}")
print("\nAs percentages:")
for term, n in counts.items():
    print(f"  {term}: {n} models ({100*n/counts.sum():.2f}%)")